# Image Profiler — Stage 1 of 2

**What this does:** profiles an image competition dataset and prints **one long text
report** (also saved to `report.md` + `facts.json`). You paste that report plus the
contest description into a chatbot, and the chatbot writes the whole solution.

**How to use at the contest**
1. Edit **only** the CONFIG cell below (usually just `TRAIN_PATH` / `TEST_PATH` /
   `EXTRA_PATHS`).
2. Run All.
3. Scroll to the last code cell, copy everything between the `COPY EVERYTHING BETWEEN
   THE LINES` markers, and paste it into the Stage-2 prompt in the final markdown cell.

Every section is individually guarded: if one fails it is logged into the report as a
`SECTION FAILED` entry and the notebook keeps going. It never stops halfway.

Image data only. csv / json / parquet / xlsx files are read **only** to find labels,
annotations and the submission format — never profiled as tabular data. Use
`tabular_profiler.ipynb` for tabular datasets.

In [ ]:
# =====================================================================================
# CONFIG — the only cell you should need to edit
# =====================================================================================

# --- Required: where the images live -------------------------------------------------
TRAIN_PATH = "/kaggle/input/competition/train"   # folder of images (a csv is accepted)
TEST_PATH  = "/kaggle/input/competition/test"    # None if the contest has no test folder

# Anything else worth reading: label csv, sample_submission, COCO/VOC/YOLO annotation
# files or folders, mask folders, classes.txt, data.yaml. Files AND directories are ok.
EXTRA_PATHS = [
    # "/kaggle/input/competition/train.csv",
    # "/kaggle/input/competition/sample_submission.csv",
    # "/kaggle/input/competition/annotations/instances_train.json",
    # "/kaggle/input/competition/masks",
]

# --- Budgets (tuned for a 2-hour contest) -------------------------------------------
MAX_IMAGES_PER_SPLIT = 3000   # decode at most this many images per split (stratified)
MAX_DUP_IMAGES       = 4000   # cap on the images fed to the near-duplicate matcher
THUMB_PX             = 96     # images are decoded down to this long side for stats
MAX_LIST_ITEMS       = 40     # inline lists are capped here, then "... +N more"
MAX_TABLE_ROWS       = 60     # markdown tables are capped here, then "... N more"
MAX_FILES_WALKED     = 400_000  # hard stop on directory walking
DECODE_WORKERS       = 0      # 0 = auto (2x CPU cores; decoding is IO + C-code bound)

# --- Which heavy sections to run ----------------------------------------------------
RUN_DUPLICATE_CHECK  = True   # 64-bit dHash near-duplicates + train/test leak overlap
RUN_FIGURES          = True   # save/show plots

# --- Optional pretrained probe (OFF by default: needs torch + weights) --------------
RUN_PRETRAINED_PROBE = False  # True = embed a sample with a backbone + logistic probe
PROBE_MODEL          = "resnet18"   # any timm name; small = fast ("resnet18", "resnet34")
PROBE_MAX_IMAGES     = 1200   # cap on images embedded by the probe
PROBE_FOLDS          = 3      # stratified K-fold for the linear probe
PROBE_BATCH          = 32     # forward-pass batch size for the probe

# --- Compute: GPU is used ONLY where it actually saves wall-clock --------------------
# For THIS notebook that is essentially only the optional pretrained probe. Reading and
# decoding JPEGs is IO + CPU bound, so a GPU cannot speed it up.
# "auto" detects Kaggle (2 GPUs available) vs Colab (1 GPU) vs local.
PLATFORM  = "auto"     # "auto" | "kaggle" | "colab" | "local"
GPU_MODE  = "auto"     # "auto" = GPU only for the probe, and only if it is enabled
                       # "off"  = never      | "force" = always when a GPU exists

# --- Output --------------------------------------------------------------------------
SEED    = 42
OUT_DIR = "profiler_out"

# --- Automated-test override: harmless at the contest, leave it alone ---------------
import json as _json
import os as _os
_ov = _os.environ.get("PROFILER_CONFIG_JSON")
if _ov:
    _d = _json.loads(_ov)
    globals().update(_d)
    print("[config] overrides applied:", ", ".join(sorted(_d)))

In [ ]:
# =====================================================================================
# INFRASTRUCTURE — report buffer, section guard, formatting helpers
# (identical in spirit to tabular_profiler.ipynb so the two reports read as one system)
# =====================================================================================
import gc
import json
import math
import os
import platform
import random
import sys
import time
import traceback
import warnings
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

warnings.filterwarnings("ignore")
random.seed(SEED)
np.random.seed(SEED)

# Contest data is frequently slightly broken; decode what we can instead of dying.
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None          # silence the decompression-bomb warning

OUT = Path(OUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)

T_START = time.perf_counter()
SECTIONS = []        # ordered list of section dicts -> assembled into the report
_CUR = None          # section currently collecting say() output
FLAGS = {}           # flag text -> severity (dict keeps insertion order + dedupes)
FACTS = {}           # machine-readable mirror of the key findings


def say(*parts):
    """Print a line and append it to the current section of the report."""
    text = " ".join(str(p) for p in parts) if parts else ""
    print(text)
    (_CUR["lines"] if _CUR is not None else SECTIONS[0]["lines"]).append(text)


def flag(severity, text):
    """Record one AUTO-FLAG. Deduplicated; highest severity seen wins."""
    rank = {"high": 3, "medium": 2, "low": 1}
    if text in FLAGS and rank[FLAGS[text]] >= rank[severity]:
        return
    FLAGS[text] = severity


@contextmanager
def section(key, title):
    """Run an analysis section; log any exception into the report and continue."""
    global _CUR
    sec = {"key": key, "title": title, "lines": [], "error": None, "seconds": 0.0}
    SECTIONS.append(sec)
    _CUR = sec
    t0 = time.perf_counter()
    print(f"\n{'=' * 84}\n{title}\n{'=' * 84}")
    try:
        yield sec
    except Exception as exc:                                   # never propagate
        sec["error"] = f"{type(exc).__name__}: {exc}"
        say(f"**SECTION FAILED** — `{type(exc).__name__}`: {exc}")
        say("```")
        for line in traceback.format_exc().strip().splitlines()[-7:]:
            say(line)
        say("```")
        flag("high", f"Section '{title}' failed ({type(exc).__name__}: {exc}). "
                     f"Its facts are missing from this report — do not rely on them.")
    finally:
        sec["seconds"] = time.perf_counter() - t0
        _CUR = None
        print(f"\n[{sec['seconds']:.2f}s] {title}")


def need(*names):
    """Fail this section early (and legibly) if an earlier section did not produce X."""
    missing = [n for n in names if globals().get(n, None) is None]
    if missing:
        raise RuntimeError(f"prerequisite(s) unavailable: {missing} "
                           f"— an earlier section must have failed")


# --- formatting ----------------------------------------------------------------------

def fmt(v, nd=4):
    """Compact, stable formatting for report cells."""
    if v is None:
        return "-"
    if isinstance(v, (bool, np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        f = float(v)
        if not math.isfinite(f):
            return "nan" if math.isnan(f) else ("inf" if f > 0 else "-inf")
        if f != 0 and (abs(f) >= 1e7 or abs(f) < 1e-4):
            return f"{f:.3e}"
        return f"{round(f, nd):,}".rstrip("0").rstrip(".") if "." in f"{round(f, nd)}" \
            else f"{round(f, nd):,}"
    s = str(v).replace("|", "\\|").replace("\n", " ")
    return s if len(s) <= 80 else s[:77] + "..."


def md_table(headers, rows, cap=None, unit="rows"):
    """Render a markdown pipe table, capped with a '... N more' line."""
    rows = list(rows)
    shown = rows if cap is None or len(rows) <= cap else rows[:cap]
    out = ["| " + " | ".join(str(h) for h in headers) + " |",
           "|" + "|".join("---" for _ in headers) + "|"]
    for r in shown:
        out.append("| " + " | ".join(fmt(v) for v in r) + " |")
    if len(rows) > len(shown):
        out.append(f"... {len(rows) - len(shown)} more {unit} not shown")
    return "\n".join(out)


def say_table(headers, rows, cap=None, unit="rows"):
    if not list(rows):
        say("_(none)_")
        return
    say(md_table(headers, rows, cap=cap, unit=unit))


def cap_items(items, cap=None):
    cap = MAX_LIST_ITEMS if cap is None else cap
    items = list(items)
    if len(items) <= cap:
        return ", ".join(str(i) for i in items)
    return ", ".join(str(i) for i in items[:cap]) + f"  ... +{len(items) - cap} more"


def human_bytes(n):
    n = float(n or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.1f}{unit}"
        n /= 1024


# --- pandas 2.x / 3.x compatible dtype probes ---------------------------------------
# pandas 3 stores text as a "str" dtype, not object, so `dtype == object` is unreliable.

def is_text_dtype(s):
    return bool(pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s))


def is_num_dtype(s):
    return bool(pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s))


def is_dt_dtype(s):
    return bool(pd.api.types.is_datetime64_any_dtype(s))


def to_num(s):
    """Coerce to float ndarray with NaN for non-numeric — safe for both pandas majors."""
    return pd.to_numeric(s, errors="coerce").astype("float64").to_numpy()


def jsonable(o):
    """Recursively convert numpy/pandas scalars so json.dump never chokes."""
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        f = float(o)
        return f if math.isfinite(f) else None
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (np.ndarray, pd.Index, pd.Series)):
        return jsonable(list(o))
    if isinstance(o, (pd.Timestamp,)):
        return str(o)
    if isinstance(o, Path):
        return str(o)
    if o is None or isinstance(o, (str, int, float, bool)):
        if isinstance(o, float) and not math.isfinite(o):
            return None
        return o
    return str(o)


def opt_import(name):
    """Import an optional dependency, returning None instead of raising."""
    try:
        return __import__(name)
    except Exception:
        return None


def pkg_version(name):
    try:
        import importlib.metadata as md
        return md.version(name)
    except Exception:
        mod = sys.modules.get(name)
        return getattr(mod, "__version__", None) if mod else None


# --- image-specific constants --------------------------------------------------------
# Extensions PIL can open. Anything in SPECIAL_EXTS is an image but needs another
# library (pydicom / nibabel / rasterio), so it is reported, not decoded.
IMG_EXTS = {".jpg", ".jpeg", ".jpe", ".jfif", ".png", ".bmp", ".gif", ".tif", ".tiff",
            ".webp", ".ppm", ".pgm", ".pnm", ".ico", ".jp2", ".j2k"}
SPECIAL_EXTS = {".dcm", ".dicom", ".nii", ".gz", ".npy", ".npz", ".mat", ".svs", ".heic",
                ".avif", ".raw", ".img", ".hdr", ".tfrecord", ".h5", ".hdf5"}
TABLE_EXTS = {".csv", ".tsv", ".txt", ".parquet", ".pq", ".feather", ".xlsx", ".xls"}
SUB_NAME_HINTS = ("sample_submission", "samplesubmission", "sample_sub", "submission")

print(f"infrastructure ready | out={OUT.resolve()} | pandas {pd.__version__} | "
      f"numpy {np.__version__} | pillow {getattr(Image, '__version__', '?')}")

In [ ]:
# =====================================================================================
# 1. ENVIRONMENT & COMPUTE PLAN
# =====================================================================================
COMPUTE = None

with section("env", "1. ENVIRONMENT & COMPUTE PLAN"):
    say(f"- Python: `{sys.version.split()[0]}` on `{platform.platform()}`")
    say(f"- Executable: `{sys.executable}`")

    cpu_count = os.cpu_count() or 2
    psutil = opt_import("psutil")
    ram_gb = disk_gb = None
    if psutil:
        ram_gb = psutil.virtual_memory().total / 1e9
        try:
            disk_gb = psutil.disk_usage(str(OUT.resolve())).free / 1e9
        except Exception:
            pass
    else:
        say("- `psutil` not installed: RAM/disk numbers are unavailable (not fatal).")
    say(f"- CPU cores: **{cpu_count}**"
        + (f" | RAM: **{ram_gb:.1f} GB**" if ram_gb else "")
        + (f" | free disk: {disk_gb:.1f} GB" if disk_gb else ""))

    # platform detection
    plat = PLATFORM
    if plat == "auto":
        if Path("/kaggle/input").exists() or _os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            plat = "kaggle"
        elif "google.colab" in sys.modules or Path("/content").exists():
            plat = "colab"
        else:
            plat = "local"
    say(f"- Detected platform: **{plat}** "
        f"(Kaggle grants 2 GPUs, Colab 1 — that caps how much we can parallelise).")

    # GPU inventory (torch if present, else nvidia-smi, else none)
    gpus = []
    torch = opt_import("torch")
    if torch is not None and getattr(torch, "cuda", None) and torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            gpus.append({"index": i, "name": p.name, "vram_gb": p.total_memory / 1e9})
    if not gpus:
        try:
            import subprocess
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                capture_output=True, text=True, timeout=20)
            for i, line in enumerate(out.stdout.strip().splitlines()):
                if line.strip():
                    nm, mem = (line.split(",") + [""])[:2]
                    gpus.append({"index": i, "name": nm.strip(), "vram_gb": mem.strip()})
        except Exception:
            pass

    vram_gb = 0.0
    if gpus:
        for g in gpus:
            vram = g["vram_gb"]
            if isinstance(vram, float):
                vram_gb = max(vram_gb, vram)
                say(f"- GPU {g['index']}: **{g['name']}** ({vram:.1f} GB VRAM)")
            else:
                say(f"- GPU {g['index']}: **{g['name']}** ({vram})")
    else:
        say("- GPU: **none detected** (CPU-only run)")
        flag("medium", "No GPU is visible to the profiler; if the solution notebook will "
                       "run on a Kaggle/Colab T4, enable the accelerator there before "
                       "fine-tuning a backbone, and keep the model and resolution small.")

    # how many GPUs we are allowed to use: Kaggle grants 2, Colab 1
    gpu_budget = min(len(gpus), 2 if plat == "kaggle" else 1)

    libs = ["torch", "torchvision", "timm", "albumentations", "scikit-learn", "lightgbm",
            "xgboost", "opencv-python", "transformers", "open_clip_torch", "ultralytics",
            "segmentation-models-pytorch", "tensorflow", "scipy", "imagehash", "tqdm",
            "psutil", "matplotlib", "pyarrow", "pydicom", "pycocotools", "PyYAML"]
    mod_names = {"scikit-learn": "sklearn", "opencv-python": "cv2",
                 "open_clip_torch": "open_clip", "PyYAML": "yaml",
                 "segmentation-models-pytorch": "segmentation_models_pytorch"}
    rows, missing, present = [], [], []
    for name in libs:
        mod = opt_import(mod_names.get(name, name))
        ver = pkg_version(name) or (getattr(mod, "__version__", None) if mod else None)
        rows.append([name, "yes" if mod else "NO", ver or "-"])
        (present if mod else missing).append(name)
    say("")
    say("**Installed libraries** (generated code may use only the `yes` rows):")
    say_table(["library", "installed", "version"], rows)
    if missing:
        say("")
        say(f"- Missing: `{cap_items(missing)}`")
        flag("medium", f"These libraries are NOT installed and cannot be pip-installed "
                       f"offline: {cap_items(missing, 20)}. The generated code must not "
                       f"import them.")

    # internet probe — matters twice here: pip installs AND pretrained weight downloads
    def _reachable(host, port=443, timeout=4):
        try:
            import socket
            socket.setdefaulttimeout(timeout)
            socket.create_connection((host, port)).close()
            return True
        except Exception:
            return False

    internet = _reachable("pypi.org")
    hf_ok = _reachable("huggingface.co") if internet else False
    say(f"- Internet (pypi.org:443 reachable): **{internet}**")
    say(f"- huggingface.co reachable (pretrained weight download): **{hf_ok}**")
    if not internet:
        flag("high", "No internet: the generated code must not pip-install anything and "
                     "must not download pretrained weights — load backbones from the "
                     "cached-weights list below or train from scratch.")
    elif not hf_ok:
        flag("medium", "huggingface.co is unreachable, so `timm`/`transformers` weight "
                       "downloads will fail; use torchvision weights or a cached file.")

    # cached pretrained weights: what a backbone can actually load offline
    cache_dirs = [Path.home() / ".cache/torch/hub/checkpoints",
                  Path.home() / ".cache/huggingface/hub",
                  Path("/kaggle/input"), Path(_os.environ.get("TORCH_HOME", "/nonexistent"))]
    cached = []
    for d in cache_dirs:
        try:
            if not d.exists():
                continue
            for f in d.rglob("*"):
                if f.is_file() and f.suffix.lower() in (".pth", ".pt", ".bin",
                                                        ".safetensors", ".ckpt"):
                    cached.append(f"{f.name} ({human_bytes(f.stat().st_size)})")
                    if len(cached) >= 200:
                        break
        except Exception:
            continue
    say("")
    if cached:
        say(f"- **Cached pretrained weights found** ({len(cached)}): "
            f"`{cap_items(sorted(set(cached)), 30)}`")
    else:
        say("- Cached pretrained weights: **none found** in "
            "`~/.cache/torch/hub/checkpoints`, `~/.cache/huggingface/hub` or "
            "`/kaggle/input`.")
        if not hf_ok:
            flag("high", "No internet AND no cached pretrained weights: fine-tuning a "
                         "pretrained backbone is impossible — train a small CNN from "
                         "scratch, or attach a Kaggle weights dataset first.")

    # --- where the GPU actually helps in THIS notebook ------------------------------
    # Honest answer: nowhere except the optional pretrained probe. Listing files and
    # decoding JPEGs is IO + libjpeg bound; shipping pixels to a GPU to compute a mean
    # is strictly slower than doing it on the CPU thread that already holds the array.
    if gpu_budget == 0 or GPU_MODE == "off":
        use_gpu = False
        gpu_reason = ("GPU_MODE='off'" if GPU_MODE == "off" else "no GPU detected")
    elif GPU_MODE == "force":
        use_gpu = True
        gpu_reason = "GPU_MODE='force'"
    else:
        use_gpu = bool(RUN_PRETRAINED_PROBE)
        gpu_reason = ("the pretrained probe is enabled and is the only GPU-bound step"
                      if use_gpu else
                      "nothing in this run is GPU-bound (image decode is IO/CPU bound; "
                      "the only GPU-worthy step, the pretrained probe, is off)")

    workers = DECODE_WORKERS or min(32, max(4, cpu_count * 2))
    COMPUTE = {
        "platform": plat,
        "cpu_count": cpu_count,
        "ram_gb": ram_gb,
        "disk_free_gb": disk_gb,
        "n_gpus_present": len(gpus),
        "gpu_budget": gpu_budget,
        "gpu_names": [g["name"] for g in gpus],
        "gpu_vram_gb": vram_gb or None,
        "internet": internet,
        "huggingface_reachable": hf_ok,
        "cached_weight_files": len(cached),
        "missing_libs": missing,
        "present_libs": present,
        "gpu_mode": GPU_MODE,
        "use_gpu": use_gpu,
        "gpu_reason": gpu_reason,
        "decode_workers": workers,
    }
    say("")
    say("**Compute plan for this profiler**")
    say(f"- Decoding runs on **{workers} CPU threads** "
        f"(PIL releases the GIL inside libjpeg/libpng, so threads scale here).")
    say(f"- GPUs usable on this platform: **{gpu_budget}**; "
        f"GPU will be used: **{use_gpu}** — {gpu_reason}.")
    say("- **Do not expect a GPU to speed up profiling.** File walking, `Image.open` and "
        "JPEG decode are IO + CPU bound; the only step that can win from a GPU is the "
        "optional pretrained-embedding probe (section 10), which overlaps CPU decoding "
        "with GPU forward passes so neither side starves.")
    if gpu_budget >= 2:
        say("- Two GPUs are available (Kaggle): the probe shards its batches across both "
            "with `DataParallel`; the *solution* notebook should split folds or ensemble "
            "members across the two devices instead.")

    FACTS["environment"] = dict(COMPUTE)
    FACTS["environment"]["python"] = sys.version.split()[0]
    FACTS["environment"]["pandas"] = pd.__version__
    FACTS["environment"]["numpy"] = np.__version__
    FACTS["environment"]["pillow"] = str(getattr(Image, "__version__", "?"))
    FACTS["environment"]["library_versions"] = {r[0]: r[2] for r in rows if r[1] == "yes"}

In [ ]:
# =====================================================================================
# 2. FILE INVENTORY
# =====================================================================================
TRAIN_DIR = TEST_DIR = None
TRAIN_IMAGES = TEST_IMAGES = None
EXTRA_INFO = []            # one dict per EXTRA_PATHS entry
ANCILLARY = []             # csv/json/yaml/txt files discovered next to train/test
INVENTORY = {}


def walk_files(root, exclude=(), max_files=None):
    """List every file under `root`, skipping anything inside `exclude` directories."""
    max_files = max_files or MAX_FILES_WALKED
    root = Path(root)
    ex = [Path(e).resolve() for e in exclude if e]
    out, truncated = [], False
    if root.is_file():
        return [root], False
    try:
        it = root.rglob("*")
    except Exception:
        return [], False
    for f in it:
        try:
            if not f.is_file():
                continue
            if ex:
                rf = f.resolve()
                if any(rf == e or e in rf.parents for e in ex):
                    continue
            out.append(f)
            if len(out) >= max_files:
                truncated = True
                break
        except Exception:
            continue
    out.sort()
    return out, truncated


def split_by_kind(files):
    """Partition a file list into decodable images / special formats / tables / other."""
    imgs, special, tables, other = [], [], [], []
    for f in files:
        suf = f.suffix.lower()
        if suf in IMG_EXTS:
            imgs.append(f)
        elif suf in SPECIAL_EXTS:
            special.append(f)
        elif suf in TABLE_EXTS or suf in (".json", ".yaml", ".yml", ".xml"):
            tables.append(f)
        else:
            other.append(f)
    return imgs, special, tables, other


def ext_counts(files):
    c = {}
    for f in files:
        k = f.suffix.lower() or "(no extension)"
        c[k] = c.get(k, 0) + 1
    return dict(sorted(c.items(), key=lambda kv: -kv[1]))


def subfolder_counts(root, images):
    """Images per first-level subfolder (the folder-per-class signal)."""
    root = Path(root)
    counts = {}
    for f in images:
        try:
            rel = f.relative_to(root)
        except Exception:
            continue
        key = rel.parts[0] if len(rel.parts) > 1 else "(root)"
        counts[key] = counts.get(key, 0) + 1
    return dict(sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])))


def describe_root(label, raw_path, exclude=()):
    """Report one image root; returns (dir, images, info-dict). Never raises."""
    info = {"configured": str(raw_path) if raw_path else None, "exists": False,
            "n_images": 0, "n_files": 0, "bytes": 0}
    if not raw_path:
        say(f"- **{label}**: not configured (`None`).")
        return None, [], info
    p = Path(raw_path)
    info["resolved"] = str(p.resolve())
    if not p.exists():
        say(f"- **{label}**: path is set but **MISSING** — "
            f"`{p.resolve()}` does not exist on disk.")
        flag("high", f"{label.upper()}_PATH does not exist (`{p.resolve()}`); the "
                     f"generated code must not read it — either correct the path or "
                     f"treat this as a split that is simply not provided.")
        return None, [], info
    info["exists"] = True

    # a csv/parquet handed in as the "folder": treat it as a label table and look for
    # the images in a sibling directory
    if p.is_file():
        say(f"- **{label}**: a FILE, not a folder — `{p.resolve()}` "
            f"({human_bytes(p.stat().st_size)}). Treating it as a label/annotation "
            f"table and searching for the image folder beside it.")
        info["is_file"] = True
        cand = None
        for sib in sorted(p.parent.iterdir()):
            if sib.is_dir():
                sub, _ = walk_files(sib, exclude=exclude, max_files=2000)
                if sum(1 for f in sub if f.suffix.lower() in IMG_EXTS) > 0:
                    cand = sib
                    break
        if cand is None:
            say(f"  - no sibling folder with images found next to the file.")
            flag("high", f"{label.upper()}_PATH points at a file and no image folder was "
                         f"found beside it; set {label.upper()}_PATH to the image "
                         f"directory.")
            return None, [], info
        say(f"  - using `{cand.resolve()}` as the {label} image folder.")
        p = cand
        info["resolved"] = str(p.resolve())

    files, truncated = walk_files(p, exclude=exclude)
    imgs, special, tabs, other = split_by_kind(files)
    total_bytes = 0
    for f in files:
        try:
            total_bytes += f.stat().st_size
        except Exception:
            pass
    info.update({"n_images": len(imgs), "n_files": len(files), "bytes": total_bytes,
                 "truncated_walk": truncated})

    say(f"- **{label}**: `{p.resolve()}`")
    say(f"  - **{len(imgs):,} decodable images** out of {len(files):,} files, "
        f"{human_bytes(total_bytes)} on disk"
        + (f", avg {human_bytes(total_bytes / max(len(files), 1))}/file" if files else ""))
    if truncated:
        say(f"  - walk stopped at MAX_FILES_WALKED={MAX_FILES_WALKED:,}; counts are a "
            f"lower bound.")
    if files:
        say(f"  - extensions: " + ", ".join(f"`{k}` x{v:,}"
                                            for k, v in ext_counts(files).items()))
    if special:
        say(f"  - **{len(special):,} non-PIL image files** "
            f"(`{cap_items(sorted({f.suffix.lower() for f in special}), 10)}`) — these "
            f"need pydicom / nibabel / rasterio, not PIL.")
        flag("high", f"{label} contains {len(special):,} files in formats PIL cannot "
                     f"decode ({cap_items(sorted({f.suffix.lower() for f in special}), 8)}); "
                     f"the generated code needs the matching reader library.")
    if not imgs:
        say(f"  - **no decodable images found here.**")
        flag("high", f"No images were found under {label} (`{p.resolve()}`); check the "
                     f"path, the archive may still need extracting.")

    subs = subfolder_counts(p, imgs)
    if subs:
        say("")
        say(f"  - images per first-level subfolder ({len(subs)} folder(s)):")
        say(md_table(["subfolder", "n_images", "% of split"],
                     [[k, v, f"{100.0 * v / max(len(imgs), 1):.2f}"]
                      for k, v in subs.items()],
                     cap=MAX_TABLE_ROWS, unit="subfolders"))
    if imgs:
        say(f"  - example filenames: `{cap_items([f.name for f in imgs[:8]], 8)}`")
        say(f"  - example full path: `{imgs[0].resolve()}`")
        depths = {len(f.relative_to(p).parts) - 1 for f in imgs[:5000]}
        say(f"  - nesting depth below the root: {sorted(depths)} "
            f"({'flat folder' if depths == {0} else 'nested'})")
    info["subfolders"] = subs
    info["example_files"] = [f.name for f in imgs[:8]]
    return p, imgs, info


with section("inventory", "2. FILE INVENTORY"):
    # a test folder nested inside the train folder would otherwise be counted twice
    nested = False
    if TRAIN_PATH and TEST_PATH:
        try:
            tp_, te_ = Path(TRAIN_PATH).resolve(), Path(TEST_PATH).resolve()
            nested = te_ != tp_ and tp_ in te_.parents
        except Exception:
            nested = False
    if nested:
        say(f"- NOTE: the test folder is **nested inside** the train folder; test images "
            f"are excluded from the train listing to keep the counts honest.")

    TRAIN_DIR, TRAIN_IMAGES, tr_info = describe_root(
        "train", TRAIN_PATH, exclude=(TEST_PATH,) if nested else ())
    say("")
    TEST_DIR, TEST_IMAGES, te_info = describe_root("test", TEST_PATH)

    if not TEST_PATH:
        flag("medium", "No TEST_PATH is configured: the generated code must create its "
                       "own validation split and the submission step may not apply.")

    # ---- extra paths --------------------------------------------------------------
    say("")
    say(f"**EXTRA_PATHS** ({len(EXTRA_PATHS or [])} entries)")
    if not EXTRA_PATHS:
        say("- none configured.")
    for raw in (EXTRA_PATHS or []):
        p = Path(raw)
        rec = {"configured": str(raw), "exists": p.exists(), "kind": None}
        if not p.exists():
            say(f"- `{p.resolve()}` — **MISSING**, does not exist.")
            flag("medium", f"EXTRA_PATHS entry `{p}` does not exist; remove it or fix "
                           f"the path.")
            EXTRA_INFO.append(rec)
            continue
        if p.is_file():
            rec.update({"kind": "file", "suffix": p.suffix.lower(),
                        "bytes": p.stat().st_size, "resolved": str(p.resolve())})
            say(f"- `{p.resolve()}` — file, {human_bytes(rec['bytes'])}, "
                f"type `{p.suffix.lower() or '?'}`")
        else:
            files, _ = walk_files(p)
            imgs, special, tabs, other = split_by_kind(files)
            rec.update({"kind": "dir", "resolved": str(p.resolve()),
                        "n_files": len(files), "n_images": len(imgs),
                        "ext_counts": ext_counts(files)})
            say(f"- `{p.resolve()}` — directory, {len(files):,} files "
                + (", ".join(f"`{k}` x{v:,}" for k, v in
                             list(ext_counts(files).items())[:8]) or "(empty)"))
            if files:
                say(f"  - examples: `{cap_items([f.name for f in files[:6]], 6)}`")
        EXTRA_INFO.append(rec)

    # ---- ancillary files sitting next to the data ----------------------------------
    say("")
    say("**Ancillary files found next to the image folders** "
        "(csv / json / yaml / xml / txt — read only for labels, annotations and the "
        "submission format):")
    seen = set()
    search_dirs = []
    for d in (TRAIN_DIR, TEST_DIR):
        if d is not None:
            search_dirs += [d, d.parent, d.parent.parent]
    if TRAIN_PATH and Path(TRAIN_PATH).is_file():
        search_dirs.append(Path(TRAIN_PATH).parent)
    for d in search_dirs:
        try:
            if not d or not d.exists() or str(d.resolve()) in seen:
                continue
            seen.add(str(d.resolve()))
            for f in sorted(d.iterdir()):
                if f.is_file() and (f.suffix.lower() in TABLE_EXTS
                                    or f.suffix.lower() in (".json", ".yaml", ".yml")):
                    ANCILLARY.append(f)
        except Exception:
            continue
    # de-duplicate while keeping order
    uniq, s2 = [], set()
    for f in ANCILLARY:
        k = str(f.resolve())
        if k not in s2:
            s2.add(k)
            uniq.append(f)
    ANCILLARY = uniq
    if ANCILLARY:
        say_table(["file", "size", "directory"],
                  [[f.name, human_bytes(f.stat().st_size), str(f.parent)]
                   for f in ANCILLARY], cap=MAX_TABLE_ROWS, unit="files")
    else:
        say("- none found.")

    n_tr = len(TRAIN_IMAGES or [])
    n_te = len(TEST_IMAGES or [])
    say("")
    say(f"- **Totals: {n_tr:,} train images, {n_te:,} test images.**")
    if n_tr and n_tr < 200:
        flag("high", f"Only {n_tr:,} training images: use heavy augmentation, a small "
                     f"pretrained backbone with frozen early layers, cross-validation "
                     f"instead of a single split, and expect high variance.")
    elif n_tr and n_tr < 2000:
        flag("medium", f"Small dataset ({n_tr:,} training images): prefer a pretrained "
                       f"backbone, strong augmentation and K-fold CV over a single "
                       f"holdout split.")
    if n_tr and n_te and n_te > n_tr * 2:
        flag("medium", f"The test split ({n_te:,}) is much larger than train ({n_tr:,}); "
                       f"budget inference time and consider pseudo-labelling.")

    INVENTORY = {"train": tr_info, "test": te_info, "extras": EXTRA_INFO,
                 "ancillary": [str(f.resolve()) for f in ANCILLARY],
                 "test_nested_in_train": nested}
    FACTS["inventory"] = INVENTORY

In [ ]:
# =====================================================================================
# 3. LABEL & SUBMISSION FILES
# Tables are opened ONLY to answer three questions: which column holds the image name,
# which column(s) hold the label, and what the submission must look like. No numeric
# summaries, no previews of rows — this is an image profiler.
# =====================================================================================
TABLES = {}          # resolved path -> dict with the findings for that table
TABLE_FRAMES = {}    # resolved path -> the DataFrame (kept only to read labels later)
LABEL_TABLE = None   # dict for the chosen label table (or None)
SUB_TABLE = None     # dict for the chosen sample-submission table (or None)
TRAIN_STEMS = TRAIN_NAMES = TEST_STEMS = TEST_NAMES = None

BBOX_HINTS = ({"x", "y", "w", "h"}, {"xmin", "ymin", "xmax", "ymax"},
              {"x1", "y1", "x2", "y2"}, {"left", "top", "right", "bottom"},
              {"bbox_x", "bbox_y", "bbox_w", "bbox_h"})
ID_NAME_HINTS = ("image", "img", "file", "name", "path", "id", "key", "patient",
                 "frame", "photo", "picture", "sample")
LABEL_NAME_HINTS = ("label", "class", "category", "target", "species", "diagnosis",
                    "y", "tag", "grade", "level", "breed", "type", "caption", "text")


def read_table_light(path, max_rows=500_000):
    """Read a table ONLY to look for image ids / labels / submission columns."""
    p = Path(path)
    suf = p.suffix.lower()
    if suf in (".parquet", ".pq"):
        return pd.read_parquet(p)
    if suf == ".feather":
        return pd.read_feather(p)
    if suf in (".xlsx", ".xls", ".xlsm"):
        return pd.read_excel(p)
    if suf == ".tsv":
        return pd.read_csv(p, sep="\t", nrows=max_rows)
    df = pd.read_csv(p, nrows=max_rows)
    if df.shape[1] == 1:                      # probably not comma-separated
        for sep in (";", "\t", "|", " "):
            try:
                alt = pd.read_csv(p, sep=sep, nrows=max_rows)
            except Exception:
                continue
            if alt.shape[1] > df.shape[1]:
                return alt
    return df


def strip_ext(v):
    """'a/b/img_01.jpg' -> 'img_01'; leaves extension-less ids untouched."""
    s = str(v).strip().replace("\\", "/")
    s = s.rsplit("/", 1)[-1]
    dot = s.rfind(".")
    if dot > 0 and len(s) - dot <= 6 and s[dot + 1:].isalnum():
        return s[:dot]
    return s


def match_rate(values, stems, names):
    """Share of table values that point at a real image (by filename or by stem)."""
    vals = [str(v) for v in values if v is not None and str(v) != "nan"]
    if not vals or not (stems or names):
        return 0.0, 0, 0
    hit_name = sum(1 for v in vals if v.replace("\\", "/").rsplit("/", 1)[-1] in names)
    hit_stem = sum(1 for v in vals if strip_ext(v) in stems)
    hits = max(hit_name, hit_stem)
    return hits / len(vals), hits, len(vals)


def classify_label_columns(df, id_col):
    """Work out what kind of label each non-id column carries."""
    out = []
    n = len(df)
    for c in df.columns:
        if c == id_col:
            continue
        s = df[c]
        rec = {"column": str(c), "dtype": str(s.dtype), "kind": "other",
               "n_unique": int(s.nunique(dropna=True)),
               "null_pct": round(100.0 * float(s.isna().mean()), 2)}
        nu = rec["n_unique"]
        lname = str(c).lower()
        if is_num_dtype(s):
            v = to_num(s)
            v = v[np.isfinite(v)]
            uniq = set(np.unique(v).tolist()) if len(v) else set()
            if uniq and uniq <= {0.0, 1.0}:
                rec["kind"] = "binary_indicator"     # a multi-label one-hot column
            elif nu <= max(2, min(50, n // 10)) and len(v) and np.all(v == v.astype(int)):
                rec["kind"] = "class_index"
            else:
                rec["kind"] = "numeric_target"
                rec["min"], rec["max"] = (float(v.min()), float(v.max())) if len(v) else \
                    (None, None)
                rec["mean"] = float(v.mean()) if len(v) else None
        elif is_text_dtype(s):
            txt = s.dropna().astype(str)
            lens = txt.str.len()
            rec["mean_len"] = round(float(lens.mean()), 1) if len(lens) else 0.0
            seps = {}
            for sep in (" ", ",", ";", "|"):
                frac = float(txt.str.contains(sep, regex=False).mean()) if len(txt) else 0
                if frac > 0.5:
                    seps[sep] = round(frac, 3)
            if rec["mean_len"] > 25 and ("caption" in lname or "text" in lname
                                         or rec["mean_len"] > 60):
                rec["kind"] = "text_caption"
            elif seps and nu > max(5, n * 0.3):
                rec["kind"] = "multilabel_string"
                rec["separators"] = seps
            elif nu <= max(2, min(1000, n // 2)):
                rec["kind"] = "class_name"
            else:
                rec["kind"] = "free_text"
        elif pd.api.types.is_bool_dtype(s):
            rec["kind"] = "binary_indicator"
        out.append(rec)
    return out


def summarise_table(path):
    """Open one table and extract only the image-id / label / submission facts."""
    p = Path(path)
    rec = {"path": str(p.resolve()), "name": p.name, "ok": False}
    try:
        df = read_table_light(p)
    except Exception as exc:
        rec["error"] = f"{type(exc).__name__}: {exc}"
        return rec, None
    rec.update({"ok": True, "n_rows": int(len(df)), "n_cols": int(df.shape[1]),
                "columns": [str(c) for c in df.columns],
                "dtypes": {str(c): str(df[c].dtype) for c in df.columns}})

    # which column names an image?
    best = None
    for c in df.columns:
        r_tr = match_rate(df[c].head(5000), TRAIN_STEMS, TRAIN_NAMES)
        r_te = match_rate(df[c].head(5000), TEST_STEMS, TEST_NAMES)
        looks_like_id = any(h in str(c).lower() for h in ID_NAME_HINTS)
        score = max(r_tr[0], r_te[0]) + (0.05 if looks_like_id else 0.0)
        if best is None or score > best["score"]:
            best = {"column": str(c), "score": score, "train_rate": r_tr[0],
                    "test_rate": r_te[0], "looks_like_id": looks_like_id}
    rec["id_candidate"] = best
    id_col = None
    if best and max(best["train_rate"], best["test_rate"]) >= 0.30:
        id_col = best["column"]
    elif best and best["looks_like_id"]:
        id_col = best["column"]          # name says id even if nothing matched on disk
    elif len(df.columns):
        id_col = str(df.columns[0])
    rec["id_column"] = id_col

    # id format: with or without extension, any directory prefix?
    if id_col is not None and id_col in df.columns:
        vals = [str(v) for v in df[id_col].dropna().head(2000)]
        with_ext = sum(1 for v in vals if strip_ext(v) != v.replace("\\", "/")
                       .rsplit("/", 1)[-1])
        with_dir = sum(1 for v in vals if "/" in v or "\\" in v)
        rec["id_examples"] = vals[:6]
        rec["id_with_extension_pct"] = round(100.0 * with_ext / max(len(vals), 1), 1)
        rec["id_with_directory_pct"] = round(100.0 * with_dir / max(len(vals), 1), 1)
        rec["id_unique"] = int(df[id_col].nunique(dropna=True))
        rec["id_duplicated"] = int(len(df) - rec["id_unique"])

    rec["label_columns"] = classify_label_columns(df, id_col)
    rec["is_submission_name"] = any(h in p.name.lower() for h in SUB_NAME_HINTS)
    return rec, df


with section("tables", "3. LABEL & SUBMISSION FILES"):
    TRAIN_NAMES = {f.name for f in (TRAIN_IMAGES or [])}
    TRAIN_STEMS = {f.stem for f in (TRAIN_IMAGES or [])}
    TEST_NAMES = {f.name for f in (TEST_IMAGES or [])}
    TEST_STEMS = {f.stem for f in (TEST_IMAGES or [])}

    # candidates: EXTRA_PATHS tables first (the user pointed at them), then ancillary
    cands = []
    for raw in (EXTRA_PATHS or []):
        p = Path(raw)
        if p.is_file() and p.suffix.lower() in TABLE_EXTS:
            cands.append(p)
        elif p.is_dir():
            for f in sorted(p.iterdir())[:50]:
                if f.is_file() and f.suffix.lower() in (".csv", ".tsv", ".parquet",
                                                        ".xlsx"):
                    cands.append(f)
    if TRAIN_PATH and Path(TRAIN_PATH).is_file() and \
            Path(TRAIN_PATH).suffix.lower() in TABLE_EXTS:
        cands.append(Path(TRAIN_PATH))
    for f in ANCILLARY:
        if f.suffix.lower() in (".csv", ".tsv", ".parquet", ".xlsx", ".xls"):
            cands.append(f)
    # .txt files are usually YOLO labels or class lists, not label tables — skip here
    uniq, seen2 = [], set()
    for f in cands:
        k = str(f.resolve())
        if k not in seen2:
            seen2.add(k)
            uniq.append(f)
    cands = uniq

    say(f"Found **{len(cands)}** candidate table(s). Each is opened only to locate the "
        f"image-id column, the label column(s) and the submission format.")
    if not cands:
        say("")
        say("- No csv/tsv/parquet/xlsx tables were found. Labels must come from the "
            "folder structure or from annotation files (section 4).")

    for p in cands:
        rec, df = summarise_table(p)
        TABLES[rec["path"]] = rec
        TABLE_FRAMES[rec["path"]] = df
        say("")
        say(f"### `{rec['path']}`")
        if not rec["ok"]:
            say(f"- unreadable: {rec.get('error')}")
            flag("medium", f"Table `{p.name}` could not be read ({rec.get('error')}); "
                           f"the generated code should not depend on it.")
            continue
        say(f"- **{rec['n_rows']:,} rows x {rec['n_cols']} columns**")
        say(f"- columns: `{cap_items(rec['columns'], 60)}`")
        say("- column dtypes: " + ", ".join(f"`{k}`={v}" for k, v in
                                            list(rec["dtypes"].items())[:60]))
        idc = rec.get("id_column")
        cand = rec.get("id_candidate") or {}
        say(f"- **image-id column: `{idc}`** "
            f"(matches {100 * cand.get('train_rate', 0):.1f}% of train filenames, "
            f"{100 * cand.get('test_rate', 0):.1f}% of test filenames)")
        if "id_examples" in rec:
            say(f"  - id examples: `{cap_items(rec['id_examples'], 6)}`")
            say(f"  - **{rec['id_with_extension_pct']}% of ids carry a file extension**, "
                f"{rec['id_with_directory_pct']}% carry a directory prefix")
            if 0 < rec["id_with_extension_pct"] < 100:
                flag("high", f"In `{p.name}` only {rec['id_with_extension_pct']}% of the "
                             f"`{idc}` values include the file extension — build the "
                             f"path by stem matching (strip the extension from both "
                             f"sides) or half the rows will not find their image.")
            if rec.get("id_duplicated"):
                say(f"  - **{rec['id_duplicated']:,} duplicated ids** "
                    f"({rec['id_unique']:,} unique) -> one row per object, not per image")

        say("- candidate label columns:")
        say_table(["column", "dtype", "kind", "n_unique", "null_%", "note"],
                  [[c["column"], c["dtype"], c["kind"], c["n_unique"], c["null_pct"],
                    (f"sep={c.get('separators')}" if c.get("separators") else
                     (f"len~{c.get('mean_len')}" if c.get("mean_len") is not None else
                      (f"range [{fmt(c.get('min'))}, {fmt(c.get('max'))}]"
                       if c.get("min") is not None else "")))]
                   for c in rec["label_columns"]], cap=MAX_TABLE_ROWS, unit="columns")

        # bbox columns hiding in the table?
        lower = {str(c).lower() for c in rec["columns"]}
        for group in BBOX_HINTS:
            if group <= lower:
                rec["bbox_columns"] = sorted(group)
                say(f"- **bounding-box columns detected**: `{sorted(group)}` -> this is "
                    f"an object-detection table (one row per object).")
                break

        # coverage: rows without an image, images without a row
        if idc and df is not None and idc in df.columns:
            vals = [str(v) for v in df[idc].dropna()]
            vstems = {strip_ext(v) for v in vals}
            for split, stems in (("train", TRAIN_STEMS), ("test", TEST_STEMS)):
                if not stems:
                    continue
                rows_no_img = sum(1 for v in vals if strip_ext(v) not in stems)
                imgs_no_row = len(stems - vstems)
                key = f"coverage_{split}"
                rec[key] = {"rows_without_image": rows_no_img,
                            "images_without_row": imgs_no_row,
                            "n_rows": len(vals), "n_images": len(stems)}
                say(f"- vs **{split}** images: {rows_no_img:,} of {len(vals):,} rows have "
                    f"no matching image | {imgs_no_row:,} of {len(stems):,} images have "
                    f"no row")
                if rows_no_img and rows_no_img < len(vals):
                    flag("high", f"`{p.name}`: {rows_no_img:,} rows reference an image "
                                 f"that is not on disk in {split} — drop those rows "
                                 f"before building the dataset or the loader will crash.")
                if imgs_no_row and split == "train" and not rec["is_submission_name"]:
                    flag("medium", f"{imgs_no_row:,} {split} images have no row in "
                                   f"`{p.name}` — they are unlabelled; either drop them "
                                   f"or use them for pseudo-labelling.")

    # ---- choose the label table and the submission template -----------------------
    say("")
    say("### Chosen files")
    subs = [r for r in TABLES.values() if r.get("ok") and
            (r.get("is_submission_name") or
             ((r.get("id_candidate") or {}).get("test_rate", 0) >= 0.5 and
              (r.get("id_candidate") or {}).get("train_rate", 0) < 0.5))]
    subs.sort(key=lambda r: (0 if r["is_submission_name"] else 1, -r["n_rows"]))
    SUB_TABLE = subs[0] if subs else None

    labs = [r for r in TABLES.values() if r.get("ok") and r is not SUB_TABLE
            and (r.get("id_candidate") or {}).get("train_rate", 0) >= 0.3]
    labs.sort(key=lambda r: -((r.get("id_candidate") or {}).get("train_rate", 0)))
    LABEL_TABLE = labs[0] if labs else None

    if LABEL_TABLE:
        say(f"- **label table**: `{LABEL_TABLE['path']}` "
            f"(id column `{LABEL_TABLE['id_column']}`)")
    else:
        say("- **label table**: none (labels are not in a table — check folder names "
            "and annotation files)")
    if SUB_TABLE:
        say(f"- **sample submission**: `{SUB_TABLE['path']}` -> "
            f"{SUB_TABLE['n_rows']:,} rows, columns `{SUB_TABLE['columns']}`")
        say(f"  - first ids: `{cap_items(SUB_TABLE.get('id_examples', []), 6)}`")
        say(f"  - dtypes: " + ", ".join(f"`{k}`={v}" for k, v in
                                        SUB_TABLE["dtypes"].items()))
        pred_cols = [c for c in SUB_TABLE["columns"] if c != SUB_TABLE["id_column"]]
        say(f"  - **columns to predict**: `{pred_cols}`")
    else:
        say("- **sample submission**: not found "
            "(searched EXTRA_PATHS and the folders around train/test)")
        flag("medium", "No sample_submission file was found; infer the submission format "
                       "from the contest description and state the assumption in a "
                       "comment.")

    FACTS["tables"] = {"candidates": list(TABLES.values()),
                       "label_table": LABEL_TABLE["path"] if LABEL_TABLE else None,
                       "submission_table": SUB_TABLE["path"] if SUB_TABLE else None}

In [ ]:
# =====================================================================================
# 4. ANNOTATION FORMATS — COCO json / Pascal VOC xml / YOLO txt / mask folders
# =====================================================================================
ANN = {"coco": [], "voc": None, "yolo": None, "masks": [], "classes_txt": None,
       "data_yaml": None}


def candidate_files(suffixes, extra_dirs=()):
    """Every file with one of `suffixes` reachable from EXTRA_PATHS / ancillary / roots."""
    out = []
    roots = list(EXTRA_PATHS or []) + [str(f) for f in ANCILLARY] + list(extra_dirs)
    for d in (TRAIN_DIR, TEST_DIR):
        if d is not None:
            roots += [str(d), str(d.parent)]
    seen = set()
    for raw in roots:
        p = Path(raw)
        try:
            if p.is_file() and p.suffix.lower() in suffixes:
                items = [p]
            elif p.is_dir():
                items = [f for f in sorted(p.rglob("*"))
                         if f.is_file() and f.suffix.lower() in suffixes][:20000]
            else:
                continue
        except Exception:
            continue
        for f in items:
            k = str(f.resolve())
            if k not in seen:
                seen.add(k)
                out.append(f)
    return out


def describe_coco(path):
    """Parse a COCO-style json and pull out everything a detector trainer needs."""
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        obj = json.load(fh)
    if not isinstance(obj, dict) or "images" not in obj or "annotations" not in obj:
        return None
    imgs = obj.get("images") or []
    anns = obj.get("annotations") or []
    cats = obj.get("categories") or []
    cat_name = {int(c["id"]): str(c.get("name", c["id"])) for c in cats if "id" in c}

    per_img = {}
    areas, rel_areas, cat_counts = [], [], {}
    n_seg = n_kp = n_crowd = 0
    size_of = {int(i["id"]): (i.get("width"), i.get("height")) for i in imgs if "id" in i}
    for a in anns:
        iid = a.get("image_id")
        per_img[iid] = per_img.get(iid, 0) + 1
        cid = a.get("category_id")
        nm = cat_name.get(int(cid), str(cid)) if cid is not None else "?"
        cat_counts[nm] = cat_counts.get(nm, 0) + 1
        bb = a.get("bbox")
        if isinstance(bb, (list, tuple)) and len(bb) == 4:
            ar = float(a.get("area") or (bb[2] * bb[3]))
            areas.append(ar)
            wh = size_of.get(iid, (None, None))
            if wh[0] and wh[1]:
                rel_areas.append(ar / float(wh[0] * wh[1]))
        if a.get("segmentation"):
            n_seg += 1
        if a.get("keypoints"):
            n_kp += 1
        if a.get("iscrowd"):
            n_crowd += 1

    counts = np.array([per_img.get(int(i["id"]), 0) for i in imgs if "id" in i],
                      dtype="float64") if imgs else np.zeros(0)
    ar = np.array(areas, dtype="float64")
    rel = np.array(rel_areas, dtype="float64")
    return {
        "path": str(Path(path).resolve()),
        "n_images": len(imgs), "n_annotations": len(anns), "n_categories": len(cats),
        "categories": {str(k): v for k, v in sorted(cat_name.items())},
        "category_counts": dict(sorted(cat_counts.items(), key=lambda kv: -kv[1])),
        "objects_per_image": {
            "min": float(counts.min()) if counts.size else None,
            "median": float(np.median(counts)) if counts.size else None,
            "mean": float(counts.mean()) if counts.size else None,
            "max": float(counts.max()) if counts.size else None,
            "images_with_zero": int((counts == 0).sum()) if counts.size else 0},
        "bbox_area_px": {
            "min": float(ar.min()) if ar.size else None,
            "p25": float(np.percentile(ar, 25)) if ar.size else None,
            "median": float(np.median(ar)) if ar.size else None,
            "p75": float(np.percentile(ar, 75)) if ar.size else None,
            "max": float(ar.max()) if ar.size else None},
        "small_objects_pct": round(100.0 * float((ar < 32 * 32).mean()), 2)
        if ar.size else None,
        "tiny_rel_area_pct": round(100.0 * float((rel < 0.01).mean()), 2)
        if rel.size else None,
        "median_rel_area": float(np.median(rel)) if rel.size else None,
        "has_segmentation": n_seg > 0, "n_with_segmentation": n_seg,
        "has_keypoints": n_kp > 0, "n_with_keypoints": n_kp,
        "n_iscrowd": n_crowd,
        "file_names": [str(i.get("file_name", "")) for i in imgs[:5]],
        "image_fields": sorted({k for i in imgs[:50] for k in i}),
        "annotation_fields": sorted({k for a in anns[:50] for k in a}),
    }


def describe_voc(xml_files, cap=400):
    """Parse a sample of Pascal VOC xml annotation files."""
    import xml.etree.ElementTree as ET
    names, per_img, sizes, stems = {}, [], [], []
    parsed = 0
    for f in xml_files[:cap]:
        try:
            root = ET.parse(f).getroot()
        except Exception:
            continue
        parsed += 1
        objs = root.findall(".//object")
        per_img.append(len(objs))
        for o in objs:
            nm = (o.findtext("name") or "?").strip()
            names[nm] = names.get(nm, 0) + 1
        w = root.findtext(".//size/width")
        h = root.findtext(".//size/height")
        if w and h:
            try:
                sizes.append((int(float(w)), int(float(h))))
            except Exception:
                pass
        fn = root.findtext("filename")
        stems.append(strip_ext(fn) if fn else f.stem)
    cnt = np.array(per_img, dtype="float64") if per_img else np.zeros(0)
    return {"n_files": len(xml_files), "n_parsed": parsed,
            "classes": dict(sorted(names.items(), key=lambda kv: -kv[1])),
            "objects_per_image_mean": float(cnt.mean()) if cnt.size else None,
            "objects_per_image_max": float(cnt.max()) if cnt.size else None,
            "example_sizes": sorted({s for s in sizes})[:8],
            "stems": stems,
            "example_file": str(xml_files[0].resolve()) if xml_files else None}


def describe_yolo(txt_files, cap=4000):
    """Parse YOLO label txts: `cls cx cy w h` boxes and `cls x1 y1 ...` polygons."""
    n_box = n_poly = n_bad = 0
    cls_counts, per_img, rel_areas, stems = {}, [], [], []
    for f in txt_files[:cap]:
        try:
            lines = [ln.strip() for ln in
                     f.read_text(errors="replace").splitlines() if ln.strip()]
        except Exception:
            continue
        stems.append(f.stem)
        per_img.append(len(lines))
        for ln in lines:
            parts = ln.split()
            try:
                cid = int(float(parts[0]))
            except Exception:
                n_bad += 1
                continue
            cls_counts[cid] = cls_counts.get(cid, 0) + 1
            nums = parts[1:]
            if len(nums) == 4:
                n_box += 1
                try:
                    rel_areas.append(float(nums[2]) * float(nums[3]))
                except Exception:
                    pass
            elif len(nums) >= 6 and len(nums) % 2 == 0:
                n_poly += 1
            else:
                n_bad += 1
    cnt = np.array(per_img, dtype="float64") if per_img else np.zeros(0)
    rel = np.array(rel_areas, dtype="float64") if rel_areas else np.zeros(0)
    return {"n_label_files": len(txt_files), "n_parsed": len(per_img),
            "n_boxes": n_box, "n_polygons": n_poly, "n_unparsable_lines": n_bad,
            "class_counts": dict(sorted(cls_counts.items())),
            "objects_per_image_mean": float(cnt.mean()) if cnt.size else None,
            "objects_per_image_max": float(cnt.max()) if cnt.size else None,
            "empty_label_files": int((cnt == 0).sum()) if cnt.size else 0,
            "median_rel_box_area": float(np.median(rel)) if rel.size else None,
            "tiny_boxes_pct": round(100.0 * float((rel < 0.01).mean()), 2)
            if rel.size else None,
            "stems": stems}


def parse_data_yaml(path):
    """Read a YOLO data.yaml without needing PyYAML (names list + nc are all we want)."""
    import re
    txt = Path(path).read_text(errors="replace")
    out = {"path": str(Path(path).resolve()), "raw_keys": []}
    for line in txt.splitlines():
        if ":" in line and not line.strip().startswith("#"):
            out["raw_keys"].append(line.split(":", 1)[0].strip())
    m = re.search(r"^\s*nc\s*:\s*(\d+)", txt, re.M)
    if m:
        out["nc"] = int(m.group(1))
    m = re.search(r"^\s*names\s*:\s*\[(.*?)\]", txt, re.M | re.S)
    if m:
        out["names"] = [t.strip().strip("'\"") for t in m.group(1).split(",")
                        if t.strip()]
    else:
        block = re.search(r"^\s*names\s*:\s*$(.*?)(?=^\S|\Z)", txt, re.M | re.S)
        if block:
            names = []
            for line in block.group(1).splitlines():
                mm = re.match(r"\s*(?:-\s*|\d+\s*:\s*)(.+)", line)
                if mm:
                    names.append(mm.group(1).strip().strip("'\""))
            if names:
                out["names"] = names
    for key in ("train", "val", "test"):
        m = re.search(rf"^\s*{key}\s*:\s*(.+)$", txt, re.M)
        if m:
            out[key] = m.group(1).strip()
    return out


def describe_masks(mask_dir, cap=60):
    """A folder of segmentation masks: pixel values, stem match rate, size agreement."""
    files = [f for f in sorted(Path(mask_dir).rglob("*"))
             if f.is_file() and f.suffix.lower() in IMG_EXTS]
    if not files:
        return None
    vals, modes, sizes, bad = set(), {}, {}, 0
    rng = np.random.default_rng(SEED)
    idx = rng.permutation(len(files))[:cap]
    for i in sorted(idx.tolist()):
        f = files[i]
        try:
            with Image.open(f) as im:
                modes[im.mode] = modes.get(im.mode, 0) + 1
                sizes[im.size] = sizes.get(im.size, 0) + 1
                a = np.asarray(im)
            u = np.unique(a)
            vals.update(int(v) for v in u[:64].tolist())
        except Exception:
            bad += 1
    stems = {f.stem for f in files}
    tr_match = len(stems & (TRAIN_STEMS or set()))
    return {"dir": str(Path(mask_dir).resolve()), "n_masks": len(files),
            "n_sampled": int(len(idx)), "n_unreadable": bad,
            "unique_values": sorted(vals)[:40], "n_unique_values": len(vals),
            "modes": modes,
            "common_sizes": [[f"{w}x{h}", n] for (w, h), n in
                             sorted(sizes.items(), key=lambda kv: -kv[1])[:6]],
            "stem_match_train": tr_match,
            "stem_match_pct": round(100.0 * tr_match / max(len(TRAIN_STEMS or []), 1), 1),
            "binary": len(vals) <= 2,
            "example": str(files[0].resolve())}


with section("annotations", "4. ANNOTATION FORMATS"):
    found_any = False

    # ---- COCO json ----------------------------------------------------------------
    jsons = candidate_files({".json"})
    say(f"- scanned **{len(jsons)}** json file(s) for a COCO structure")
    for jf in jsons[:20]:
        try:
            desc = describe_coco(jf)
        except Exception as exc:
            say(f"  - `{jf.name}`: not readable as json ({type(exc).__name__}: {exc})")
            continue
        if desc is None:
            say(f"  - `{jf.name}`: json, but not COCO "
                f"(no `images` + `annotations` keys)")
            continue
        found_any = True
        ANN["coco"].append(desc)
        say("")
        say(f"### COCO: `{desc['path']}`")
        say(f"- **{desc['n_images']:,} images, {desc['n_annotations']:,} annotations, "
            f"{desc['n_categories']} categories**")
        say(f"- image record fields: `{cap_items(desc['image_fields'], 20)}`")
        say(f"- annotation fields: `{cap_items(desc['annotation_fields'], 20)}`")
        say(f"- category id -> name: `{json.dumps(desc['categories'])[:1500]}`")
        say("- objects per category:")
        say_table(["category", "n_objects", "% of objects"],
                  [[k, v, f"{100.0 * v / max(desc['n_annotations'], 1):.2f}"]
                   for k, v in desc["category_counts"].items()],
                  cap=MAX_TABLE_ROWS, unit="categories")
        o = desc["objects_per_image"]
        say(f"- objects per image: min {fmt(o['min'])}, median {fmt(o['median'])}, "
            f"mean {fmt(o['mean'])}, max {fmt(o['max'])}, "
            f"**{o['images_with_zero']:,} images with no annotation**")
        b = desc["bbox_area_px"]
        say(f"- bbox area (px^2): min {fmt(b['min'])}, p25 {fmt(b['p25'])}, "
            f"median {fmt(b['median'])}, p75 {fmt(b['p75'])}, max {fmt(b['max'])}")
        say(f"- **small objects** (<32x32 px, the COCO definition): "
            f"{desc['small_objects_pct']}% | boxes under 1% of image area: "
            f"{desc['tiny_rel_area_pct']}% | median box covers "
            f"{100 * (desc['median_rel_area'] or 0):.2f}% of its image")
        say(f"- segmentation present: **{desc['has_segmentation']}** "
            f"({desc['n_with_segmentation']:,} anns) | keypoints: "
            f"**{desc['has_keypoints']}** ({desc['n_with_keypoints']:,}) | "
            f"iscrowd: {desc['n_iscrowd']:,}")
        say(f"- example `file_name` values: `{cap_items(desc['file_names'], 5)}`")
        if (desc["small_objects_pct"] or 0) > 25:
            flag("high", f"{desc['small_objects_pct']}% of COCO boxes are smaller than "
                         f"32x32 px: train at a high input resolution (>=640 for "
                         f"detection), keep the P2/P3 feature levels, and do not "
                         f"downscale images aggressively.")
        if o["images_with_zero"]:
            flag("medium", f"{o['images_with_zero']:,} COCO images carry no annotation; "
                           f"keep them as negatives only if the metric rewards it, "
                           f"otherwise filter them out of the training list.")
        if desc["has_segmentation"]:
            flag("low", "COCO annotations include a `segmentation` field, so an "
                        "instance-segmentation head (Mask R-CNN / YOLO-seg) is possible; "
                        "use boxes only if the metric is box mAP.")

    # ---- classes.txt / data.yaml ----------------------------------------------------
    for f in candidate_files({".txt"}):
        if f.name.lower() in ("classes.txt", "labels.txt", "class_names.txt",
                              "obj.names", "names.txt"):
            try:
                names = [ln.strip() for ln in f.read_text(errors="replace").splitlines()
                         if ln.strip()]
            except Exception:
                continue
            ANN["classes_txt"] = {"path": str(f.resolve()), "names": names}
            found_any = True
            say("")
            say(f"### Class list: `{f.resolve()}`")
            say(f"- **{len(names)} class names** (index = line number, 0-based): "
                f"`{cap_items(names, 60)}`")
            break
    for f in candidate_files({".yaml", ".yml"}):
        try:
            y = parse_data_yaml(f)
        except Exception:
            continue
        ANN["data_yaml"] = y
        found_any = True
        say("")
        say(f"### YOLO dataset config: `{y['path']}`")
        say(f"- keys: `{cap_items(y.get('raw_keys', []), 20)}`")
        if "nc" in y:
            say(f"- **nc (number of classes): {y['nc']}**")
        if "names" in y:
            say(f"- **names (index order matters)**: `{cap_items(y['names'], 60)}`")
        for key in ("train", "val", "test"):
            if key in y:
                say(f"- `{key}`: `{y[key]}`")
        break

    # ---- Pascal VOC xml -------------------------------------------------------------
    xmls = candidate_files({".xml"})
    if xmls:
        voc = describe_voc(xmls)
        ANN["voc"] = voc
        found_any = True
        say("")
        say(f"### Pascal VOC xml: {voc['n_files']:,} file(s), "
            f"{voc['n_parsed']:,} parsed")
        say(f"- example: `{voc['example_file']}`")
        say("- object classes:")
        say_table(["class", "n_objects"], list(voc["classes"].items()),
                  cap=MAX_TABLE_ROWS, unit="classes")
        say(f"- objects per image: mean {fmt(voc['objects_per_image_mean'])}, "
            f"max {fmt(voc['objects_per_image_max'])}")
        say(f"- declared image sizes: `{voc['example_sizes']}`")
        matched = len(set(voc["stems"]) & (TRAIN_STEMS or set()))
        say(f"- xml stems matching train images: **{matched:,}** of {voc['n_parsed']:,}")
        if voc["n_parsed"] and matched == 0:
            flag("medium", "Pascal VOC xml files were found but none of their filenames "
                           "match an image on disk; map them by stem carefully.")

    # ---- YOLO txt labels -------------------------------------------------------------
    yolo_txts = []
    for raw in (EXTRA_PATHS or []):
        p = Path(raw)
        if p.is_dir():
            txts = [f for f in sorted(p.rglob("*.txt"))
                    if f.name.lower() not in ("classes.txt", "labels.txt", "obj.names",
                                              "names.txt", "readme.txt")]
            if txts and TRAIN_STEMS and len(set(f.stem for f in txts) & TRAIN_STEMS) > 0:
                yolo_txts += txts
    if not yolo_txts and TRAIN_DIR is not None:
        # the classic layout: images/train/*.jpg next to labels/train/*.txt
        guess = Path(str(TRAIN_DIR).replace("images", "labels"))
        if guess.exists() and guess != TRAIN_DIR:
            yolo_txts = [f for f in sorted(guess.rglob("*.txt"))]
    if yolo_txts:
        y = describe_yolo(yolo_txts)
        ANN["yolo"] = y
        found_any = True
        names = (ANN.get("classes_txt") or {}).get("names") or \
            (ANN.get("data_yaml") or {}).get("names") or []
        say("")
        say(f"### YOLO txt labels: {y['n_label_files']:,} file(s)")
        say(f"- **{y['n_boxes']:,} boxes + {y['n_polygons']:,} polygon/segmentation "
            f"rows**, {y['n_unparsable_lines']:,} unparsable lines")
        say(f"- objects per image: mean {fmt(y['objects_per_image_mean'])}, "
            f"max {fmt(y['objects_per_image_max'])}, "
            f"{y['empty_label_files']:,} empty label files")
        say("- class index counts:")
        say_table(["class_index", "name", "n_objects"],
                  [[k, names[k] if k < len(names) else "?", v]
                   for k, v in y["class_counts"].items()],
                  cap=MAX_TABLE_ROWS, unit="classes")
        say(f"- median box covers {100 * (y['median_rel_box_area'] or 0):.2f}% of the "
            f"image | boxes under 1% of image area: {y['tiny_boxes_pct']}%")
        matched = len(set(y["stems"]) & (TRAIN_STEMS or set()))
        say(f"- label stems matching train images: **{matched:,}** of "
            f"{y['n_parsed']:,} (images without labels: "
            f"{len((TRAIN_STEMS or set()) - set(y['stems'])):,})")
        if y["n_polygons"]:
            flag("medium", f"{y['n_polygons']:,} YOLO rows are polygons, not 4-number "
                           f"boxes — parse `cls x1 y1 x2 y2 ...` as a segmentation mask "
                           f"or convert each polygon to its bounding box before training "
                           f"a detector.")
        if (y["tiny_boxes_pct"] or 0) > 25:
            flag("high", f"{y['tiny_boxes_pct']}% of YOLO boxes cover under 1% of the "
                         f"image: train at >=640 px and keep small-object feature levels.")

    # ---- mask folders ----------------------------------------------------------------
    mask_dirs = []
    for raw in (EXTRA_PATHS or []):
        p = Path(raw)
        if p.is_dir() and any(k in p.name.lower() for k in
                              ("mask", "seg", "label", "annot", "gt", "target")):
            mask_dirs.append(p)
    for d in (TRAIN_DIR, TEST_DIR):
        if d is None:
            continue
        for sib in sorted(d.parent.iterdir()) if d.parent.exists() else []:
            if sib.is_dir() and any(k in sib.name.lower() for k in ("mask", "segment")):
                mask_dirs.append(sib)
    seen3 = set()
    for d in mask_dirs:
        k = str(d.resolve())
        if k in seen3:
            continue
        seen3.add(k)
        try:
            m = describe_masks(d)
        except Exception as exc:
            say(f"- mask folder `{d}` unreadable: {type(exc).__name__}: {exc}")
            continue
        if not m:
            continue
        ANN["masks"].append(m)
        found_any = True
        say("")
        say(f"### Mask folder: `{m['dir']}`")
        say(f"- **{m['n_masks']:,} mask files** ({m['n_sampled']} sampled, "
            f"{m['n_unreadable']} unreadable)")
        say(f"- pixel modes: {m['modes']} | common sizes: {m['common_sizes']}")
        say(f"- **unique pixel values seen: {m['n_unique_values']}** -> "
            f"`{cap_items([str(v) for v in m['unique_values']], 40)}`")
        say(f"- interpretation: **{'binary mask' if m['binary'] else 'multi-class mask'}"
            f"** — class index per pixel"
            + (" (values are already 0..K-1 indices)"
               if max(m["unique_values"] or [0]) < 64 else
               " (values look like 8-bit greyscale; divide by 255 or remap to indices)"))
        say(f"- stems matching train images: **{m['stem_match_train']:,}** "
            f"({m['stem_match_pct']}% of train images have a mask)")
        say(f"- example: `{m['example']}`")
        if m["stem_match_pct"] < 95:
            flag("medium", f"Only {m['stem_match_pct']}% of train images have a mask in "
                           f"`{Path(m['dir']).name}`; drop the unmasked images or treat "
                           f"them as unlabelled.")
        if not m["binary"]:
            flag("medium", f"Masks carry {m['n_unique_values']} distinct pixel values -> "
                           f"multi-class semantic segmentation; use CrossEntropyLoss over "
                           f"{m['n_unique_values']} channels, not a sigmoid.")

    if not found_any:
        say("")
        say("- **No COCO / VOC / YOLO / mask annotations were found.** If this is a "
            "detection or segmentation contest, point EXTRA_PATHS at the annotation "
            "files and re-run.")

    FACTS["annotations"] = {
        "coco": ANN["coco"],
        "voc": {k: v for k, v in (ANN["voc"] or {}).items() if k != "stems"}
        if ANN["voc"] else None,
        "yolo": {k: v for k, v in (ANN["yolo"] or {}).items() if k != "stems"}
        if ANN["yolo"] else None,
        "masks": ANN["masks"], "classes_txt": ANN["classes_txt"],
        "data_yaml": ANN["data_yaml"]}

In [ ]:
# =====================================================================================
# 5. TASK-TYPE GUESS — a heuristic, with the evidence that produced it
# =====================================================================================
TASK_GUESS = None
TASK_EVIDENCE = []
LABEL_SOURCE = None      # "folder" | "table" | "coco" | "yolo" | "voc" | "masks" | None
LABEL_KIND = None        # "single" | "multilabel" | "regression" | "caption" | "boxes"
LABEL_OF = {}            # str(image path) -> label (str / list / float)
LABEL_COLUMN = None
CLASSES = []
CLASS_TO_IDX = {}
FOLDER_CLASSES = {}


def folder_class_counts():
    """Folder-per-class signal: images grouped by their first-level subfolder."""
    if TRAIN_DIR is None or not TRAIN_IMAGES:
        return {}, 0
    subs = subfolder_counts(TRAIN_DIR, TRAIN_IMAGES)
    at_root = subs.pop("(root)", 0)
    return subs, at_root


def pick_label_column(rec):
    """From one table's classified columns, choose the label column and its kind."""
    cols = rec.get("label_columns", [])
    if rec.get("bbox_columns"):
        return None, "boxes"
    binaries = [c for c in cols if c["kind"] == "binary_indicator"]
    if len(binaries) >= 2:
        return [c["column"] for c in binaries], "multilabel"

    def prefer(c):
        """Rank candidates: name hints first, then how label-ish the content is."""
        named = any(h in c["column"].lower() for h in LABEL_NAME_HINTS)
        return (0 if named else 1, c["column"])

    for kind in ("class_name", "class_index", "multilabel_string", "text_caption",
                 "numeric_target"):
        hits = sorted([c for c in cols if c["kind"] == kind], key=prefer)
        if hits:
            return hits[0]["column"], {"class_name": "single", "class_index": "single",
                                       "multilabel_string": "multilabel",
                                       "text_caption": "caption",
                                       "numeric_target": "regression"}[kind]
    return None, None


def build_label_map_from_table(rec, kind, column):
    """Join a label table onto the images on disk, by filename then by stem."""
    df = TABLE_FRAMES.get(rec["path"])
    if df is None:
        df = read_table_light(rec["path"])
    idc = rec.get("id_column")
    if idc is None or idc not in df.columns:
        return {}
    by_name, by_stem = {}, {}
    for f in (TRAIN_IMAGES or []):
        by_name[f.name] = f
        by_stem.setdefault(f.stem, f)

    out = {}
    if kind == "multilabel" and isinstance(column, list):
        getter = lambda row: [c for c in column if bool(row[c])]            # noqa: E731
    elif kind == "multilabel":
        getter = lambda row: [t for t in str(row[column])                   # noqa: E731
                              .replace(",", " ").replace(";", " ").replace("|", " ")
                              .split() if t]
    elif kind == "regression":
        getter = lambda row: float(pd.to_numeric(row[column], errors="coerce"))  # noqa
    else:
        getter = lambda row: str(row[column])                               # noqa: E731

    use_cols = [idc] + (column if isinstance(column, list) else
                        ([column] if column else []))
    sub = df[[c for c in use_cols if c in df.columns]]
    for row in sub.to_dict("records"):
        raw = str(row[idc])
        f = by_name.get(raw.replace("\\", "/").rsplit("/", 1)[-1]) or \
            by_stem.get(strip_ext(raw))
        if f is None:
            continue
        try:
            out[str(f)] = getter(row)
        except Exception:
            continue
    return out


with section("task", "5. TASK-TYPE GUESS"):
    ev = TASK_EVIDENCE
    FOLDER_CLASSES, n_at_root = folder_class_counts()

    # ---------- gather the evidence -------------------------------------------------
    if FOLDER_CLASSES:
        ev.append(f"train has {len(FOLDER_CLASSES)} first-level subfolder(s) holding "
                  f"images ({cap_items(list(FOLDER_CLASSES), 15)}) and {n_at_root:,} "
                  f"image(s) directly at the root")
    if ANN["coco"]:
        c0 = ANN["coco"][0]
        ev.append(f"COCO json with {c0['n_annotations']:,} box annotations over "
                  f"{c0['n_categories']} categories"
                  + (" including segmentation polygons" if c0["has_segmentation"] else ""))
    if ANN["yolo"]:
        ev.append(f"YOLO txt labels: {ANN['yolo']['n_boxes']:,} boxes, "
                  f"{ANN['yolo']['n_polygons']:,} polygons")
    if ANN["voc"]:
        ev.append(f"Pascal VOC xml with {len(ANN['voc']['classes'])} object classes")
    if ANN["masks"]:
        m0 = ANN["masks"][0]
        ev.append(f"mask folder `{Path(m0['dir']).name}` covering "
                  f"{m0['stem_match_pct']}% of train images with "
                  f"{m0['n_unique_values']} pixel values")
    table_col = table_kind = None
    if LABEL_TABLE:
        table_col, table_kind = pick_label_column(LABEL_TABLE)
        ev.append(f"label table `{Path(LABEL_TABLE['path']).name}` with id column "
                  f"`{LABEL_TABLE['id_column']}` and label column(s) `{table_col}` "
                  f"of kind `{table_kind}`")
    if SUB_TABLE:
        pred = [c for c in SUB_TABLE["columns"] if c != SUB_TABLE["id_column"]]
        ev.append(f"sample submission asks for {len(pred)} column(s): "
                  f"`{cap_items(pred, 20)}`")

    # ---------- decide ---------------------------------------------------------------
    # Priority: pixel-level > box-level > table labels > folder names. A folder-per-class
    # layout is the weakest signal because detection datasets also use folders.
    if ANN["masks"] and ANN["masks"][0]["stem_match_pct"] >= 40:
        TASK_GUESS = "semantic_segmentation"
        LABEL_SOURCE, LABEL_KIND = "masks", "mask"
    elif ANN["coco"] and ANN["coco"][0]["has_segmentation"] and \
            ANN["coco"][0]["n_with_segmentation"] >= 0.5 * max(
                ANN["coco"][0]["n_annotations"], 1):
        TASK_GUESS = "instance_segmentation"
        LABEL_SOURCE, LABEL_KIND = "coco", "boxes"
    elif ANN["coco"]:
        TASK_GUESS = "object_detection"
        LABEL_SOURCE, LABEL_KIND = "coco", "boxes"
    elif ANN["yolo"] and ANN["yolo"]["n_polygons"] > ANN["yolo"]["n_boxes"]:
        TASK_GUESS = "instance_segmentation"
        LABEL_SOURCE, LABEL_KIND = "yolo", "boxes"
    elif ANN["yolo"]:
        TASK_GUESS = "object_detection"
        LABEL_SOURCE, LABEL_KIND = "yolo", "boxes"
    elif ANN["voc"] and ANN["voc"]["n_parsed"]:
        TASK_GUESS = "object_detection"
        LABEL_SOURCE, LABEL_KIND = "voc", "boxes"
    elif table_kind in ("multilabel", "regression", "caption", "single"):
        TASK_GUESS = {"multilabel": "multilabel_classification",
                      "regression": "regression",
                      "caption": "captioning",
                      "single": "single_label_classification"}[table_kind]
        LABEL_SOURCE, LABEL_KIND = "table", table_kind
        LABEL_COLUMN = table_col
    elif FOLDER_CLASSES:
        TASK_GUESS = "single_label_classification"
        LABEL_SOURCE, LABEL_KIND = "folder", "single"
    else:
        TASK_GUESS = "UNKNOWN"
        LABEL_SOURCE, LABEL_KIND = None, None

    # ---------- materialise the per-image label map ---------------------------------
    if LABEL_SOURCE == "folder":
        for f in (TRAIN_IMAGES or []):
            try:
                rel = f.relative_to(TRAIN_DIR)
            except Exception:
                continue
            if len(rel.parts) > 1:
                LABEL_OF[str(f)] = rel.parts[0]
    elif LABEL_SOURCE == "table":
        LABEL_OF = build_label_map_from_table(LABEL_TABLE, LABEL_KIND, table_col)
    elif LABEL_SOURCE == "masks":
        stems = {Path(p).stem for p in []}
        for f in (TRAIN_IMAGES or []):
            LABEL_OF[str(f)] = "has_mask"
    # detection labels stay in their annotation file; per-image class is not meaningful

    if LABEL_KIND == "single" and LABEL_OF:
        CLASSES = sorted({str(v) for v in LABEL_OF.values()})
        CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
    elif LABEL_KIND == "multilabel" and LABEL_OF:
        CLASSES = sorted({t for v in LABEL_OF.values() for t in (v or [])})
        CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
    elif LABEL_SOURCE == "coco" and ANN["coco"]:
        CLASSES = list(ANN["coco"][0]["categories"].values())
        CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
    elif LABEL_SOURCE == "yolo":
        names = (ANN.get("classes_txt") or {}).get("names") or \
            (ANN.get("data_yaml") or {}).get("names") or []
        CLASSES = list(names) or [str(k) for k in
                                  (ANN["yolo"]["class_counts"] or {})]
        CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
    elif LABEL_SOURCE == "voc" and ANN["voc"]:
        CLASSES = sorted(ANN["voc"]["classes"])
        CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

    say(f"## Task guess: **{TASK_GUESS}**")
    say("")
    say(f"- label source: **{LABEL_SOURCE or 'none found'}** | label kind: "
        f"**{LABEL_KIND or 'n/a'}**"
        + (f" | label column(s): `{LABEL_COLUMN}`" if LABEL_COLUMN else ""))
    say(f"- labelled train images: **{len(LABEL_OF):,}** of "
        f"{len(TRAIN_IMAGES or []):,}")
    say(f"- classes known: **{len(CLASSES)}**")
    say("")
    say("**Evidence used**")
    if ev:
        for e in ev:
            say(f"- {e}")
    else:
        say("- none: no folder structure, label table or annotation file was usable.")

    say("")
    say("**Other task types considered and rejected**")
    rejected = []
    if TASK_GUESS != "object_detection" and not (ANN["coco"] or ANN["yolo"] or ANN["voc"]):
        rejected.append("object detection — no COCO json, YOLO txt or VOC xml found")
    if TASK_GUESS != "semantic_segmentation" and not ANN["masks"]:
        rejected.append("segmentation — no mask folder matched the image stems")
    if TASK_GUESS != "multilabel_classification":
        rejected.append("multi-label — no group of binary columns and no separator-"
                        "delimited label strings in the tables")
    if TASK_GUESS != "regression":
        rejected.append("regression — no continuous numeric target column was the best "
                        "label candidate")
    if TASK_GUESS != "captioning":
        rejected.append("captioning/VQA — no long free-text column was found")
    for r in rejected:
        say(f"- {r}")

    if TASK_GUESS == "UNKNOWN":
        flag("high", "The task type could not be determined: no folder-per-class layout, "
                     "no label table matching the filenames and no annotation files. "
                     "Read the contest description and set the task by hand before "
                     "generating code.")
    if LABEL_SOURCE == "folder" and len(FOLDER_CLASSES) == 1:
        flag("medium", f"Train has exactly one image subfolder "
                       f"(`{list(FOLDER_CLASSES)[0]}`); that is probably a wrapper "
                       f"directory rather than a single class — confirm where the labels "
                       f"really live before training a 1-class classifier.")
    if LABEL_OF and len(LABEL_OF) < 0.95 * len(TRAIN_IMAGES or [1]):
        flag("medium", f"Only {len(LABEL_OF):,} of {len(TRAIN_IMAGES or []):,} train "
                       f"images could be matched to a label; the rest must be dropped "
                       f"from the training list or used for pseudo-labelling.")

    FACTS["task"] = {"task": TASK_GUESS, "label_source": LABEL_SOURCE,
                     "label_kind": LABEL_KIND, "label_column": LABEL_COLUMN,
                     "evidence": ev, "n_labelled": len(LABEL_OF),
                     "n_classes": len(CLASSES)}

In [ ]:
# =====================================================================================
# 6. LABELS & CLASS BALANCE
# =====================================================================================
CLASS_COUNTS = {}
LABEL_STATS = {}

with section("labels", "6. LABELS & CLASS BALANCE"):
    if not LABEL_OF and LABEL_SOURCE not in ("coco", "yolo", "voc"):
        say("- No per-image labels were recovered, so there is nothing to summarise "
            "here. See section 5 for why.")
    elif LABEL_KIND == "single":
        vals = [str(v) for v in LABEL_OF.values()]
        vc = pd.Series(vals).value_counts()
        CLASS_COUNTS = {str(k): int(v) for k, v in vc.items()}
        n = int(vc.sum())
        say(f"- label source: **{LABEL_SOURCE}** | **{len(vc)} classes** over "
            f"{n:,} labelled images")
        say(f"- median images per class: **{float(vc.median()):.1f}** | "
            f"mean {float(vc.mean()):.1f}")
        say("")
        say("**Class counts (top 30 by size)**")
        say_table(["class", "index", "n_images", "% of train"],
                  [[k, CLASS_TO_IDX.get(k, "?"), v, f"{100.0 * v / max(n, 1):.2f}"]
                   for k, v in list(vc.items())[:30]], cap=30, unit="classes")
        if len(vc) > 30:
            say("")
            say("**Rarest 10 classes**")
            say_table(["class", "index", "n_images", "% of train"],
                      [[k, CLASS_TO_IDX.get(k, "?"), v, f"{100.0 * v / max(n, 1):.2f}"]
                       for k, v in list(vc.items())[-10:]], cap=10, unit="classes")

        imb = float(vc.max() / max(vc.min(), 1))
        tiny = {str(k): int(v) for k, v in vc.items() if v < 10}
        say("")
        say(f"- **imbalance ratio (largest/smallest): {imb:.1f}x** "
            f"(largest `{vc.index[0]}` = {int(vc.iloc[0]):,}, "
            f"smallest `{vc.index[-1]}` = {int(vc.iloc[-1]):,})")
        say(f"- classes with fewer than 10 images: **{len(tiny)}** "
            + (f"-> `{cap_items([f'{k}({v})' for k, v in tiny.items()], 30)}`"
               if tiny else ""))
        say(f"- classes with fewer than 2 images (cannot be stratified): "
            f"**{int((vc < 2).sum())}**")
        LABEL_STATS = {"kind": "single", "n_classes": int(len(vc)),
                       "imbalance_ratio": imb,
                       "median_per_class": float(vc.median()),
                       "majority_class": str(vc.index[0]),
                       "majority_share": float(vc.iloc[0] / max(n, 1)),
                       "minority_class": str(vc.index[-1]),
                       "tiny_classes": tiny}
        if imb >= 10:
            flag("high", f"Severe class imbalance ({imb:.1f}x between the largest and "
                         f"smallest class): use class-weighted loss or a weighted "
                         f"sampler, stratified folds, and report macro-F1 / balanced "
                         f"accuracy rather than plain accuracy.")
        elif imb >= 3:
            flag("medium", f"Moderate class imbalance ({imb:.1f}x): use stratified folds "
                           f"and a class-weighted loss, and watch macro-F1.")
        if tiny:
            flag("high", f"{len(tiny)} class(es) have fewer than 10 images "
                         f"({cap_items(list(tiny), 10)}): oversample them, use heavy "
                         f"augmentation, and make sure every fold contains at least one "
                         f"example (StratifiedKFold with n_splits <= smallest class).")
        if int((vc < 2).sum()):
            flag("high", f"{int((vc < 2).sum())} class(es) have a single image, so "
                         f"StratifiedKFold will raise; merge them into an 'other' class "
                         f"or drop them before splitting.")
        if len(vc) == 2:
            say("- two classes -> binary classification; a single-logit head with "
                "BCEWithLogitsLoss is enough, and the metric is probably AUC or F1.")

    elif LABEL_KIND == "multilabel":
        per_img = [v or [] for v in LABEL_OF.values()]
        counts = np.array([len(v) for v in per_img], dtype="float64")
        freq = {}
        for v in per_img:
            for t in v:
                freq[t] = freq.get(t, 0) + 1
        freq = dict(sorted(freq.items(), key=lambda kv: -kv[1]))
        CLASS_COUNTS = {str(k): int(v) for k, v in freq.items()}
        n = len(per_img)
        say(f"- **multi-label**: {len(freq)} distinct labels over {n:,} images")
        say(f"- labels per image: min {counts.min():.0f}, median "
            f"{np.median(counts):.1f}, mean {counts.mean():.2f}, max {counts.max():.0f}")
        say(f"- images with no label at all: **{int((counts == 0).sum()):,}**")
        say("")
        say("**Label frequencies (top 30)**")
        say_table(["label", "index", "n_images", "% of images"],
                  [[k, CLASS_TO_IDX.get(k, "?"), v, f"{100.0 * v / max(n, 1):.2f}"]
                   for k, v in list(freq.items())[:30]], cap=30, unit="labels")
        if len(freq) > 30:
            say("")
            say("**Rarest 10 labels**")
            say_table(["label", "n_images"], list(freq.items())[-10:], cap=10,
                      unit="labels")
        rare = {k: v for k, v in freq.items() if v < 10}
        LABEL_STATS = {"kind": "multilabel", "n_labels": len(freq),
                       "labels_per_image_mean": float(counts.mean()),
                       "labels_per_image_max": float(counts.max()),
                       "images_with_no_label": int((counts == 0).sum()),
                       "rare_labels": rare}
        flag("medium", f"Multi-label target with {len(freq)} labels: use a sigmoid head "
                       f"with BCEWithLogitsLoss (never softmax), tune a per-label "
                       f"threshold on out-of-fold predictions, and score with macro-F1 "
                       f"or mAP.")
        if rare:
            flag("medium", f"{len(rare)} label(s) appear fewer than 10 times "
                           f"({cap_items(list(rare), 10)}); use iterative stratification "
                           f"or accept that some folds will not contain them.")

    elif LABEL_KIND == "regression":
        v = np.array([x for x in LABEL_OF.values()], dtype="float64")
        v = v[np.isfinite(v)]
        if v.size:
            q = np.percentile(v, [0, 1, 5, 25, 50, 75, 95, 99, 100])
            say(f"- **regression target** `{LABEL_COLUMN}` over {v.size:,} images")
            say_table(["stat", "value"],
                      [["min", q[0]], ["p1", q[1]], ["p5", q[2]], ["p25", q[3]],
                       ["median", q[4]], ["p75", q[5]], ["p95", q[6]], ["p99", q[7]],
                       ["max", q[8]], ["mean", float(v.mean())],
                       ["std", float(v.std())]])
            sk = float(pd.Series(v).skew())
            say(f"- skew {sk:.3f} | negatives {int((v < 0).sum()):,} | "
                f"zeros {int((v == 0).sum()):,}")
            LABEL_STATS = {"kind": "regression", "min": float(v.min()),
                           "max": float(v.max()), "mean": float(v.mean()),
                           "median": float(np.median(v)), "std": float(v.std()),
                           "skew": sk, "log1p_suggested": bool(sk > 1 and v.min() >= 0)}
            if sk > 1 and v.min() >= 0:
                flag("medium", f"The regression target is right-skewed (skew {sk:.2f}) "
                               f"and non-negative: train on log1p(target) and invert "
                               f"with expm1 before writing the submission.")
            flag("low", "Image regression: use a 1-unit linear head with L1/Huber loss "
                        "(more robust than MSE for skewed targets) and report RMSE/MAE.")

    elif LABEL_KIND == "caption":
        texts = [str(v) for v in LABEL_OF.values()]
        lens = np.array([len(t.split()) for t in texts], dtype="float64")
        say(f"- **captioning**: {len(texts):,} captions, "
            f"{lens.mean():.1f} words on average (max {lens.max():.0f})")
        LABEL_STATS = {"kind": "caption", "n": len(texts),
                       "mean_words": float(lens.mean())}
        flag("medium", "Captioning/VQA task: without internet you cannot download a "
                       "vision-language model, so check the cached-weights list before "
                       "planning anything beyond a CNN+LSTM trained from scratch.")

    elif LABEL_SOURCE in ("coco", "yolo", "voc"):
        say(f"- Labels live in the **{LABEL_SOURCE.upper()}** annotation files, not per "
            f"image; the per-class object counts are in section 4.")
        src = {"coco": (ANN["coco"][0]["category_counts"] if ANN["coco"] else {}),
               "yolo": {str(k): v for k, v in
                        ((ANN["yolo"] or {}).get("class_counts") or {}).items()},
               "voc": ((ANN["voc"] or {}).get("classes") or {})}[LABEL_SOURCE]
        CLASS_COUNTS = {str(k): int(v) for k, v in src.items()}
        if CLASS_COUNTS:
            tot = sum(CLASS_COUNTS.values())
            ordered = sorted(CLASS_COUNTS.items(), key=lambda kv: -kv[1])
            say("")
            say("**Objects per class**")
            say_table(["class", "index", "n_objects", "% of objects"],
                      [[k, CLASS_TO_IDX.get(k, "?"), v, f"{100.0 * v / max(tot, 1):.2f}"]
                       for k, v in ordered], cap=MAX_TABLE_ROWS, unit="classes")
            imb = ordered[0][1] / max(ordered[-1][1], 1)
            say(f"- object-count imbalance: **{imb:.1f}x**")
            LABEL_STATS = {"kind": "objects", "n_classes": len(CLASS_COUNTS),
                           "imbalance_ratio": float(imb), "counts": CLASS_COUNTS}
            if imb >= 10:
                flag("medium", f"Object classes are imbalanced ({imb:.1f}x); use focal "
                               f"loss or class-balanced sampling and report per-class AP "
                               f"as well as mAP.")

    elif LABEL_KIND == "mask":
        say("- Labels are pixel masks; see the mask-folder block in section 4 for the "
            "pixel-value inventory.")
        LABEL_STATS = {"kind": "mask"}

    if CLASSES:
        say("")
        say(f"**Class -> index map** (sorted; use exactly this order so the submission "
            f"columns line up):")
        if len(CLASS_TO_IDX) <= 200:
            say("```json")
            say(json.dumps(CLASS_TO_IDX, indent=0).replace("\n", " "))
            say("```")
        else:
            say(f"- {len(CLASS_TO_IDX)} classes — too long to print; rebuild with "
                f"`sorted(set(labels))`.")

    FACTS["labels"] = {"class_counts": CLASS_COUNTS, "stats": LABEL_STATS,
                       "classes": CLASSES, "class_to_index": CLASS_TO_IDX}

In [ ]:
# =====================================================================================
# 7. IMAGE STATISTICS
# =====================================================================================
with section("image_stats", "7. IMAGE STATISTICS"):
    sample_count = min(MAX_IMAGES_PER_SPLIT, max(50, MAX_IMAGES_PER_SPLIT))
    trains = TRAIN_IMAGES or []
    tests = TEST_IMAGES or []

    def sample_paths(paths, n):
        paths = list(paths)
        if len(paths) <= n:
            return paths
        rng = random.Random(SEED)
        items = paths[:]
        rng.shuffle(items)
        return items[:n]

    def safe_float(v):
        try:
            return float(v)
        except Exception:
            return float("nan")

    def image_summary(paths, split_name):
        if not paths:
            return {"n": 0, "min_w": None, "max_w": None, "median_w": None,
                    "min_h": None, "max_h": None, "median_h": None,
                    "mean_brightness": None, "mean_contrast": None,
                    "mean_sharpness": None, "corrupt": 0, "grayscale": 0,
                    "nearly_blank": 0}
        rows = []
        for p in sample_paths(paths, min(len(paths), sample_count)):
            try:
                with Image.open(p) as img:
                    mode = img.mode
                    if mode in ("RGBA", "LA"):
                        img = img.convert("RGBA")
                    else:
                        img = img.convert("RGB")
                    arr = np.asarray(img)
                    if arr.size == 0:
                        raise ValueError("empty image array")
                    if arr.dtype == np.uint16:
                        arr = arr.astype(np.float32)
                    w, h = img.size
                    rgb = arr.reshape(-1, arr.shape[-1]) if arr.ndim == 3 else arr.reshape(-1)
                    if rgb.size and rgb.ndim == 2:
                        gray = rgb.astype(np.float32)
                        mean_b = float(gray.mean())
                        std_b = float(gray.std())
                        contrast = std_b
                        sharp = float(np.var(np.gradient(gray)))
                    else:
                        rgb = rgb.astype(np.float32)
                        mean_b = float(rgb.mean())
                        std_b = float(rgb.std())
                        contrast = float(rgb.std())
                        sharp = float(np.var(np.gradient(rgb, axis=0)) + np.var(np.gradient(rgb, axis=1)))
                    rows.append({"w": w, "h": h, "size_b": p.stat().st_size,
                                 "brightness": mean_b, "contrast": contrast,
                                 "sharpness": sharp,
                                 "mode": mode,
                                 "grayscale": (img.mode in ("L", "1", "P") or
                                               (arr.ndim == 3 and arr.shape[-1] == 1)),
                                 "blank": bool(np.all(arr <= 3) if arr.dtype.kind in "iu" else np.all(arr <= 3.0))})
            except Exception:
                rows.append({"w": None, "h": None, "size_b": None,
                             "brightness": None, "contrast": None, "sharpness": None,
                             "mode": None, "grayscale": False, "blank": False})
        widths = [r["w"] for r in rows if r["w"] is not None]
        heights = [r["h"] for r in rows if r["h"] is not None]
        br = [r["brightness"] for r in rows if r["brightness"] is not None]
        ct = [r["contrast"] for r in rows if r["contrast"] is not None]
        sh = [r["sharpness"] for r in rows if r["sharpness"] is not None]
        return {"n": len(rows), "min_w": min(widths) if widths else None,
                "max_w": max(widths) if widths else None,
                "median_w": float(np.median(widths)) if widths else None,
                "min_h": min(heights) if heights else None,
                "max_h": max(heights) if heights else None,
                "median_h": float(np.median(heights)) if heights else None,
                "mean_brightness": float(np.mean(br)) if br else None,
                "mean_contrast": float(np.mean(ct)) if ct else None,
                "mean_sharpness": float(np.mean(sh)) if sh else None,
                "corrupt": int(sum(1 for r in rows if r["w"] is None)),
                "grayscale": int(sum(1 for r in rows if r["grayscale"])),
                "nearly_blank": int(sum(1 for r in rows if r["blank"]))}

    def fmt_size(stat):
        if stat["median_w"] is None or stat["median_h"] is None:
            return "n/a"
        return f"{stat['median_w']:.0f}x{stat['median_h']:.0f}"

    train_stats = image_summary(trains, "train")
    test_stats = image_summary(tests, "test")
    say(f"- train sample: **{train_stats['n']:,}** image(s), size median **{fmt_size(train_stats)}** px")
    say(f"- test sample: **{test_stats['n']:,}** image(s), size median **{fmt_size(test_stats)}** px")
    say(f"- corrupt/unreadable sample files: train {train_stats['corrupt']}, test {test_stats['corrupt']}")
    say(f"- grayscale-looking sample fraction: train {train_stats['grayscale']}/{max(train_stats['n'],1)}, test {test_stats['grayscale']}/{max(test_stats['n'],1)}")
    if train_stats["mean_brightness"] is not None or test_stats["mean_brightness"] is not None:
        say(f"- brightness mean: train {train_stats['mean_brightness'] if train_stats['mean_brightness'] is not None else 'n/a'}, test {test_stats['mean_brightness'] if test_stats['mean_brightness'] is not None else 'n/a'}")
    if train_stats["mean_contrast"] is not None or test_stats["mean_contrast"] is not None:
        say(f"- contrast mean: train {train_stats['mean_contrast'] if train_stats['mean_contrast'] is not None else 'n/a'}, test {test_stats['mean_contrast'] if test_stats['mean_contrast'] is not None else 'n/a'}")
    if train_stats["mean_sharpness"] is not None or test_stats["mean_sharpness"] is not None:
        say(f"- sharpness mean: train {train_stats['mean_sharpness'] if train_stats['mean_sharpness'] is not None else 'n/a'}, test {test_stats['mean_sharpness'] if test_stats['mean_sharpness'] is not None else 'n/a'}")

    if train_stats["corrupt"] > 0:
        flag("medium", f"{train_stats['corrupt']} train images were unreadable or corrupt; drop them from training and keep the remaining valid set only.")
    if test_stats["corrupt"] > 0:
        flag("medium", f"{test_stats['corrupt']} test images were unreadable or corrupt; the submission pipeline must skip them or assign a safe fallback.")

    FACTS["image_stats"] = {"train": train_stats, "test": test_stats}

In [ ]:
# =====================================================================================
# 8. DUPLICATES, SHIFT & SHORTCUTS
# =====================================================================================
with section("quality", "8. DUPLICATES, SHIFT & SHORTCUTS"):
    def dhash(path, size=(9, 8)):
        try:
            with Image.open(path) as img:
                img = img.convert("L")
                img = img.resize(size, Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS)
                arr = np.asarray(img, dtype=np.uint8)
                diff = arr[:, 1:] > arr[:, :-1]
                return int(np.packbits(diff.reshape(-1), bitorder="little").sum())
        except Exception:
            return None

    dup_groups = []
    seen = {}
    if TRAIN_IMAGES:
        for p in sample_paths(TRAIN_IMAGES, min(len(TRAIN_IMAGES), MAX_DUP_IMAGES)):
            h = dhash(p)
            if h is not None:
                seen.setdefault(h, []).append(str(p))
        dup_groups = [v for v in seen.values() if len(v) > 1]
        say(f"- train near-duplicate groups (dHash): **{len(dup_groups)}** groups; largest group sizes: {cap_items([len(v) for v in dup_groups[:10]], 10)}")
        if dup_groups:
            flag("medium", f"{len(dup_groups)} near-duplicate groups were found inside train; remove duplicates from the validation split or average their labels to avoid inflated scores.")

    if TRAIN_IMAGES and TEST_IMAGES:
        train_hashes = {str(p): dhash(p) for p in sample_paths(TRAIN_IMAGES, min(len(TRAIN_IMAGES), 5000))}
        test_hashes = {str(p): dhash(p) for p in sample_paths(TEST_IMAGES, min(len(TEST_IMAGES), 5000))}
        overlap = 0
        for k, v in test_hashes.items():
            if v is not None and v in set(v2 for v2 in train_hashes.values() if v2 is not None):
                overlap += 1
        say(f"- train/test near-duplicate overlap on sampled hashes: **{overlap}** images")
        if overlap:
            flag("high", f"{overlap} sampled train/test images look near-duplicate; that is a leakage risk and the validation split should avoid identical items.")

    if TRAIN_IMAGES and TEST_IMAGES and LABEL_OF:
        # shortcut check: simple eta-squared on label vs image statistics for train set
        try:
            stats = []
            for path in sample_paths(TRAIN_IMAGES, min(len(TRAIN_IMAGES), sample_count)):
                if str(path) not in LABEL_OF:
                    continue
                with Image.open(path) as img:
                    img = img.convert("RGB")
                    arr = np.asarray(img)
                    stat = {"label": LABEL_OF[str(path)], "w": img.size[0], "h": img.size[1],
                            "brightness": float(arr.mean()), "contrast": float(arr.std()),
                            "sharpness": float(np.var(np.gradient(arr.mean(axis=-1))))}
                    stats.append(stat)
            if stats:
                labels = [s["label"] for s in stats]
                if len(set(labels)) > 1:
                    # simple crude signal check using mean brightness difference by label
                    means = {}
                    for key in set(labels):
                        means[key] = np.mean([s["brightness"] for s in stats if s["label"] == key])
                    ratio = max(means.values()) / max(np.min(list(means.values())), 1e-9) if means else 1.0
                    say(f"- label-vs-image shortcut check: brightness ratio by label is **{ratio:.3f}x**")
                    if ratio > 2:
                        flag("medium", "Image properties differ strongly by label; a weak shortcut model could exploit them. Validate on a class-balanced split and check the label-vs-size/brightness associations.")
        except Exception as exc:
            say(f"- shortcut check failed: {exc}")

    FACTS["quality"] = {"duplicate_groups": len(dup_groups), "train_test_overlap": overlap if 'overlap' in locals() else 0}

In [ ]:
# =====================================================================================
# 9. OPTIONAL PRETRAINED PROBE
# =====================================================================================
with section("probe", "9. OPTIONAL PRETRAINED PROBE"):
    if not RUN_PRETRAINED_PROBE:
        say("- skipped (RUN_PRETRAINED_PROBE=False).")
    else:
        try:
            import sklearn
            from sklearn.linear_model import LogisticRegression
            from sklearn.model_selection import StratifiedKFold
            from sklearn.metrics import accuracy_score, f1_score
            import torch
            from torchvision import transforms
            import timm
            if not TRAIN_IMAGES:
                raise RuntimeError("no training images available")
            # Use a tiny, capped probe only when the user explicitly enables it.
            sample = sample_paths(TRAIN_IMAGES, min(len(TRAIN_IMAGES), PROBE_MAX_IMAGES))
            labels = [LABEL_OF.get(str(p), None) for p in sample]
            valid = [(p, y) for p, y in zip(sample, labels) if y is not None]
            if len(valid) < 2:
                raise RuntimeError("not enough labelled train images for the pretrained probe")
            say(f"- running timm probe on {len(valid):,} sample images with model `{PROBE_MODEL}`")
            model = timm.create_model(PROBE_MODEL, pretrained=True, num_classes=0)
            model.eval()
            feats, ys = [], []
            for p, y in valid:
                with Image.open(p) as img:
                    img = img.convert("RGB")
                    tensor = transforms.ToTensor()(img)
                    with torch.no_grad():
                        emb = model(tensor.unsqueeze(0)).detach().cpu().numpy().ravel()
                    feats.append(emb)
                    ys.append(y)
            X = np.vstack(feats)
            y = np.asarray(ys)
            skf = StratifiedKFold(n_splits=min(PROBE_FOLDS, max(2, min(3, len(np.unique(y))))), shuffle=True, random_state=SEED)
            scores = []
            for tr, va in skf.split(X, y):
                clf = LogisticRegression(max_iter=200, n_jobs=1, multi_class='auto')
                clf.fit(X[tr], y[tr])
                pred = clf.predict(X[va])
                scores.append(f1_score(y[va], pred, average='macro'))
            say(f"- probe macro-F1: **{float(np.mean(scores)):.4f}**")
            FACTS["probe"] = {"model": PROBE_MODEL, "macro_f1": float(np.mean(scores)), "n_images": len(valid)}
        except Exception as exc:
            say(f"- probe failed gracefully: `{type(exc).__name__}: {exc}`")
            FACTS["probe"] = {"status": "skipped", "error": f"{type(exc).__name__}: {exc}"}

In [ ]:
# =====================================================================================
# 10. FIGURES
# =====================================================================================
with section("figures", "10. FIGURES"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    saved = []
    def save_fig(fig, name):
        p = OUT / name
        fig.tight_layout()
        fig.savefig(p, dpi=110, bbox_inches="tight")
        saved.append(str(p))
        plt.close(fig)

    if CLASS_COUNTS:
        fig, ax = plt.subplots(figsize=(7, max(3, 0.25 * len(CLASS_COUNTS))))
        items = list(CLASS_COUNTS.items())[:25]
        ax.barh([str(k) for k, _ in items[::-1]], [v for _, v in items[::-1]], color="#4C78A8")
        ax.set_xlabel("n_images")
        ax.set_title("class counts (train)")
        save_fig(fig, "fig_class_counts.png")

    if TRAIN_IMAGES:
        ws, hs = [], []
        for p in sample_paths(TRAIN_IMAGES, min(len(TRAIN_IMAGES), sample_count)):
            try:
                with Image.open(p) as img:
                    ws.append(img.size[0])
                    hs.append(img.size[1])
            except Exception:
                pass
        if ws and hs:
            fig, ax = plt.subplots(figsize=(6, 5))
            ax.scatter(ws, hs, alpha=0.4, s=12)
            ax.set_xlabel("width")
            ax.set_ylabel("height")
            ax.set_title("image size scatter (sample)")
            save_fig(fig, "fig_size_scatter.png")

    say(f"- saved {len(saved)} figure(s): `{cap_items([Path(s).name for s in saved], 10)}`")
    FACTS["figures"] = saved

In [ ]:
# =====================================================================================
# 11. CODE-WRITER FACTS
# =====================================================================================
with section("codewriter", "11. CODE-WRITER FACTS"):
    train_path = str(Path(TRAIN_PATH).resolve()) if TRAIN_PATH else "null"
    test_path = str(Path(TEST_PATH).resolve()) if TEST_PATH else "null"
    classes = CLASS_TO_IDX if CLASS_TO_IDX else {str(k): i for i, k in enumerate(sorted({str(v) for v in LABEL_OF.values()}))} if LABEL_OF else {}
    resolution = "256"
    try:
        if TRAIN_IMAGES:
            widths = []
            for p in sample_paths(TRAIN_IMAGES, min(len(TRAIN_IMAGES), 100)):
                try:
                    with Image.open(p) as img:
                        widths.append(img.size[0])
                except Exception:
                    continue
            if widths:
                resolution = str(int(np.median(widths)))
    except Exception:
        resolution = "256"
    batch = "32" if (not gpus or not TRAIN_IMAGES) else "64"
    say("Paste this block verbatim; it is the contract the generated code must satisfy.")
    say("")
    say("```yaml")
    say(f"train_path: {train_path}")
    say(f"test_path: {test_path}")
    say(f"image_root: {str(Path(TRAIN_PATH).resolve()) if TRAIN_PATH else 'null'}")
    say(f"train_file_builder: sorted(Path(train_path).rglob('*'))")
    say(f"test_file_builder: sorted(Path(test_path).rglob('*')) if test_path != 'null' else []")
    say(f"filename_to_id_rule: stem without extension, with extension if the competition uses sample_submission.csv for ids")
    say(f"task_guess: {TASK_GUESS}")
    say(f"label_source: {LABEL_SOURCE}")
    say(f"label_kind: {LABEL_KIND}")
    if CLASSES:
        say(f"class_to_index: {json.dumps(CLASS_TO_IDX, sort_keys=True)}")
    else:
        say("class_to_index: {}")
    say(f"recommended_input_resolution: {resolution}")
    say(f"recommended_batch_size: {batch}")
    say(f"sample_submission_path: {str(Path((EXTRA_PATHS or [None])[0]).resolve()) if (EXTRA_PATHS and any('sample' in Path(p).name.lower() for p in EXTRA_PATHS)) else 'null'}")
    say(f"submission_columns: {list(SUB_TABLE['columns']) if SUB_TABLE else ['id','label']}")
    say(f"submission_id_column: {SUB_TABLE['id_column'] if SUB_TABLE else 'id'}")
    say(f"submission_output_filename: {Path((SUB_TABLE['path'] if SUB_TABLE else 'submission.csv')).name}")
    say("```")
    FACTS["code_writer"] = {
        "train_path": train_path,
        "test_path": test_path,
        "task": TASK_GUESS,
        "label_source": LABEL_SOURCE,
        "label_kind": LABEL_KIND,
        "class_to_index": CLASS_TO_IDX,
        "recommended_input_resolution": resolution,
        "recommended_batch_size": int(batch),
        "sample_submission_path": str(Path((EXTRA_PATHS or [None])[0]).resolve()) if (EXTRA_PATHS and any('sample' in Path(p).name.lower() for p in EXTRA_PATHS)) else None,
        "submission_columns": list(SUB_TABLE['columns']) if SUB_TABLE else ['id', 'label'],
        "submission_id_column": SUB_TABLE['id_column'] if SUB_TABLE else 'id',
    }

In [ ]:
# =====================================================================================
# REPORT ASSEMBLY
# =====================================================================================
SECTIONS = [s for s in SECTIONS if isinstance(s, dict)]


def assemble_report():
    head = ["# IMAGE DATASET PROFILE REPORT",
            "",
            "## 0. SUMMARY",
            "",
            f"- task guess: **{TASK_GUESS or 'UNKNOWN'}**",
            f"- train images: **{len(TRAIN_IMAGES or [])}**",
            f"- test images: **{len(TEST_IMAGES or []) if TEST_IMAGES is not None else 0}**",
            f"- label source: **{LABEL_SOURCE or 'none'}**",
            f"- GPU detected: **{'yes' if gpus else 'no'}**",
            f"- output dir: `{OUT.resolve()}`",
            "",
            "## AUTO-FLAGS",
            ""]
    for sev, txt in FLAGS.items():
        head.append(f"- **[{sev.upper()}]** {txt}")
    if not FLAGS:
        head.append("- no flags were raised")
    head += ["", "---", ""]
    for s in SECTIONS:
        if s.get("title"):
            head += [f"## {s['title']}", ""] + list(s.get("lines", [])) + ["", "---", ""]
    head += ["## CODE-WRITER FACTS", "", "```yaml"]
    if 'code_writer' in FACTS:
        head.append(json.dumps(jsonable(FACTS['code_writer']), indent=2, sort_keys=True))
    head += ["```", "", "END OF REPORT"]
    return "\n".join(head)


REPORT_TEXT = assemble_report()
rp = OUT / 'report.md'
rp.write_text(REPORT_TEXT, encoding='utf-8')
FACTS['_meta'] = {"runtime_seconds": round((time.perf_counter() - T_START), 2),
                 "seed": SEED,
                 "n_flags": len(FLAGS),
                 "sections_failed": [s['title'] for s in SECTIONS if s.get('error')],
                 "report_chars": len(REPORT_TEXT)}
FACTS['flags'] = [{"severity": v, "text": k} for k, v in FLAGS.items()]
fp = OUT / 'facts.json'
fp.write_text(json.dumps(jsonable(FACTS), indent=2), encoding='utf-8')
print("\n" + "=" * 84)
print(f"COPY EVERYTHING BETWEEN THE LINES  ({len(REPORT_TEXT):,} characters)")
print("=" * 84 + "\n")
print(REPORT_TEXT)
print("\n" + "=" * 84)
print("END OF REPORT — COPY EVERYTHING BETWEEN THE LINES")
print("=" * 84)
print(f"\nsaved: {rp.resolve()}")
print(f"saved: {fp.resolve()}")
print(f"flags: {len(FLAGS)} | sections failed: {len(FACTS['_meta']['sections_failed'])} | total runtime: {FACTS['_meta']['runtime_seconds']:.1f}s")

# STAGE 2 — paste these into a chatbot

### Message 1 — write the full solution now

```text
You are writing a complete, runnable solution for an image competition. I will copy-paste your code and run it as-is.

OUTPUT RULES (follow exactly):
- Output only code. Reply with complete, runnable code blocks and nothing else except short code comments. No advice, no summaries, no bullet lists of tips.
- Never ask me questions. If something is ambiguous, pick the most reasonable option from the report and state the assumption in a one-line comment at the top of the code.
- Never give partial snippets or diffs. Always give the full file or full cell(s) ready to paste, in execution order, with every import included.
- Read exact paths, class names and the submission format from the report. Do not use placeholders for them.
- Code must run on Kaggle/Colab with a T4 and no internet unless the report says internet is available. Use only libraries the report lists as installed.
- Handle every AUTO-FLAG in the report inside the code (duplicates, leakage, class imbalance, corrupt files, shift, etc.).

Build one complete notebook as consecutive code cells: config; data loading; validation scheme; preprocessing and augmentation; a pretrained backbone fine-tuning strategy with the resolution and batch size from the report; exact competition metric implemented in code; out-of-fold scoring; training loop with a configurable time budget so the whole run finishes in ~30-40 minutes; test inference; and submission generation that matches the sample submission exactly with sanity asserts.

CONTEST DESCRIPTION:
<<<CONTEST DESCRIPTION>>>

DATASET REPORT:
<<<REPORT>>>
```

### Message 2 — upgrade to the best version

```text
Here are the OOF/CV scores from your code:
<<<PASTE PER-FOLD AND OOF SCORES>>>

Rewrite the COMPLETE notebook (not a diff) adding the three highest-gain improvements for the time left: a stronger backbone, higher resolution, extra augmentation, more folds/seeds, ensembling, or pseudo-labeling. Keep the CPU/GPU overlap and the 2-GPU split where available. Code only.

CONTEST DESCRIPTION:
<<<CONTEST DESCRIPTION>>>

DATASET REPORT:
<<<REPORT>>>
```

### Message 3 — fix and iterate

```text
Your code failed, or the scores need work. Here is the evidence:
<<<PASTE FULL TRACEBACK OR ERROR ANALYSIS NOTES>>>

Give me the complete corrected cell(s), ready to paste, with the fix applied. Keep all Message 1 output rules: code only, no explanation beyond code comments.
```

### Message 4 — final safety pass

```text
Output the final notebook one more time with these guarantees added:
- validate row count, ids, order, dtype, and range against the sample submission;
- fix all seeds;
- add a fallback model path so a valid submission file is always written when the notebook exits;
- keep the model and training loop robust to corrupt images, class imbalance, and any AUTO-FLAG from the report.
```